<div style="background:linear-gradient(135deg,#00553A 0%,#00704A 55%,#00A86A 100%);border-radius:20px;padding:38px 40px 34px 40px;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#F5C242;font-size:12px;letter-spacing:3px;font-weight:700;">AFRICAN DEVELOPMENT BANK &nbsp;·&nbsp; AU STATAFRIC &nbsp;·&nbsp; STG17</div><div style="color:#fff;font-size:2.15em;font-weight:800;margin-top:10px;line-height:1.12;">Night-Time Lights&nbsp;: data collection<br>NASA Black Marble (VNP46)</div><div style="color:#E6F6EE;font-size:1.05em;font-style:italic;margin-top:14px;max-width:62em;line-height:1.6;">A reproducible acquisition chain, driven by a single parameter&nbsp;— the country's ISO3 code&nbsp;— which queries the NASA&nbsp;CMR catalogue, tests the four Black Marble products over a fixed grid of years, then downloads and verifies the selected granules.</div><div style="margin-top:22px;height:4px;width:130px;background:#F5C242;border-radius:2px;"></div><div style="color:#CFEEDE;font-size:.92em;margin-top:16px;">Technical workshop <b>&ldquo;Emerging Issues, Emerging Practice&rdquo;</b> &nbsp;·&nbsp; Day 4 — Hands-on, part 1 &ldquo;Collect&rdquo;<br>Runs on <b>Google Colab</b>, <b>Kaggle</b> and <b>local Jupyter</b> (Windows, macOS, Linux)</div></div>

## What this notebook does

You fill in **one parameter** — the country's ISO3 code (`CIV`, `SDN`, `RWA`, `TUN`…) — and the notebook does the rest: it derives the country's geographic extent, computes the VIIRS tiles that cover it, queries NASA's catalogue for the four Black Marble products over a fixed grid of years, builds an availability inventory, then downloads a verified sample.

| # | Step | Network | NASA credentials |
|---|---|---|---|
| 1 | Detect the runtime environment and set up working folders | no | no |
| 2 | Check and install dependencies | pip | no |
| 3 | Retrieve the Earthdata token (4 possible sources) | no | — |
| 4 | Collection parameters: **country, products, temporal grid** | no | no |
| 5 | National extent and the matching VIIRS tiles | no | no |
| 6 | Query functions for the NASA CMR catalogue | no | no |
| 7 | **Availability test across the four products** | yes | no |
| 8 | Resumable, verified download | yes | **yes** |
| 9 | Quality control: open an HDF5 file and preview it | no | no |

Steps 1 to 7 run **without any credentials**: searching NASA's catalogue is public. The token is only needed when files are actually downloaded, at step 8. You can therefore run most of the notebook, and all of the teaching material, before you have even created an account.

<div style="background:#F4F7F5;border:1px solid #00704A44;border-radius:12px;padding:13px 18px;margin:10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><b style="color:#00704A;">🔑 How to use it, in three moves</b><div style="color:#231F20;margin-top:5px;line-height:1.55;"><b>1.</b> Run the cells in order (<i>Runtime → Run all</i>). <b>2.</b> Edit the step 4 cell only — that is the single place to change. <b>3.</b> Rerun. The notebook is <i>idempotent</i>: files already downloaded and valid are skipped, and interrupted downloads resume where they stopped.</div></div>

---

## Background: what is Black Marble data?

The **VIIRS/DNB** sensor (*Day–Night Band*), flying on the Suomi-NPP and NOAA-20 satellites, measures the light emitted towards space from the Earth's surface every night. NASA turns those raw measurements into a corrected product suite called **Black Marble** (the VNP46 collection), in which the effects of the Moon, the atmosphere, clouds and snow have been removed. The unit is the **nW·cm⁻²·sr⁻¹** (nanowatt per square centimetre per steradian).

Four products differ only in their **temporal frequency** — they share the same grid, the same tiles and the same file structure:

| Product | Frequency | One file covers | Main variable | Typical use |
|---|---|---|---|---|
| `VNP46A1` | daily | one night, raw at-sensor radiance | `DNB_At_Sensor_Radiance_500m` | diagnostics, checking the source data |
| `VNP46A2` | daily | one night, lunar-BRDF corrected and gap-filled | `Gap_Filled_DNB_BRDF-Corrected_NTL` | crisis monitoring, power outages |
| `VNP46A3` | monthly | one calendar month (composite) | `NearNadir_Composite_Snow_Free` | seasonality, sub-annual series |
| `VNP46A4` | annual | one calendar year (composite) | `NearNadir_Composite_Snow_Free` | long series, regional comparisons |

**The tile rule.** The globe is cut into fixed 10° × 10° cells in latitude/longitude, indexed `h` (column, 0 to 35, west to east) and `v` (row, 0 to 17, north to south). Each file corresponds to **one tile and one date**. A country therefore maps to a fixed set of tiles: Côte d'Ivoire fits inside `h17v07` and `h17v08`, Sudan needs six. Step 5 computes that set automatically from the country's extent.

**The naming rule.** A granule is called, for example, `VNP46A4.A2023001.h17v08.001.2024031102344.h5`, that is `product . A + year + day-of-year . tile . version . processing timestamp . h5`. Since the processing timestamp is **impossible to guess**, you never build a URL by hand: you query the catalogue, which returns the real names.

<div style="background:#E8F5EF;border:1px solid #00704A44;border-radius:12px;padding:13px 18px;margin:10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><b style="color:#00704A;">ℹ️ Why go through the CMR catalogue rather than the LAADS directory</b><div style="color:#231F20;margin-top:5px;line-height:1.55;">The <b>CMR</b> (<i>Common Metadata Repository</i>) is NASA's unified search engine. One request is enough to ask for <i>&ldquo;every granule of product X that intersects this extent between these two dates&rdquo;</i> — the catalogue does the geographic intersection for you, and it returns each file's size as a bonus, which lets you estimate the volume <b>before</b> downloading anything. The search is public and needs no token.</div></div>

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">STEP 01 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Runtime environment</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Colab, Kaggle or local — the notebook adapts on its own</div></div>

The one genuinely delicate point of portability is **file paths**. A hard-coded `C:/NTL_LOCAL` survives neither Colab, nor Kaggle, nor a colleague on macOS. The cell below detects the platform, picks a writable working root, and actually tests it before keeping it.

You can force your own folder by setting the `NTL_HOME` environment variable before launching Jupyter.

In [ ]:
# =============================================================================
#  STEP 1 - Runtime environment, working folders, console encoding
# =============================================================================
import os, sys, platform, shutil, tempfile, datetime as dt
from pathlib import Path

# --- 1.1  Console: force UTF-8 where possible --------------------------------
# On Windows, a legacy cp1252 console crashes on a plain print("✓").
# We try to reconfigure stdout; if that fails we fall back to ASCII symbols.
# The notebook stays readable everywhere.
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass


def _symbols():
    table = {"ok": "\u2713", "ko": "\u2717", "att": "\u26a0", "fl": "\u2192", "pt": "\u2022"}
    try:
        "".join(table.values()).encode(getattr(sys.stdout, "encoding", None) or "utf-8")
        return table
    except Exception:
        return {"ok": "[OK]", "ko": "[X]", "att": "[!]", "fl": "->", "pt": "-"}


SYM = _symbols()
WIDTH = 78


def header(text):
    """Print a fixed-width section header. No dependency."""
    print("\n" + "=" * WIDTH)
    print("  " + text.upper())
    print("=" * WIDTH)


def field(key, value, symbol=" "):
    print(f"  {symbol} {key:<30} {value}")


# --- 1.2  Which platform? ----------------------------------------------------
def detect_platform():
    """Return 'Colab', 'Kaggle' or 'Local'. Test order matters."""
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists():
        return "Kaggle"
    if "google.colab" in sys.modules or Path("/content").exists():
        return "Colab"
    return "Local"


PLATFORM = detect_platform()


# --- 1.3  A working root that is genuinely writable --------------------------
def choose_root(platform):
    """Try several locations and return the first one we can actually write to."""
    candidates = []
    if os.environ.get("NTL_HOME"):
        candidates.append(Path(os.environ["NTL_HOME"]))
    if platform == "Colab":
        candidates.append(Path("/content/NTL_LOCAL"))
    elif platform == "Kaggle":
        candidates.append(Path("/kaggle/working/NTL_LOCAL"))
    candidates += [Path.home() / "NTL_LOCAL",
                  Path.cwd() / "NTL_LOCAL",
                  Path(tempfile.gettempdir()) / "NTL_LOCAL"]

    for c in candidates:
        try:
            c.mkdir(parents=True, exist_ok=True)
            probe = c / ".test_ecriture"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return c
        except Exception:
            continue
    raise RuntimeError("No writable folder found. Set NTL_HOME.")


ROOT = choose_root(PLATFORM)
FOLDERS = {
    "h5_cache":  ROOT / "h5_cache",     # downloaded .h5 granules
    "catalogue": ROOT / "catalogue",    # the CSV inventories produced here
    "figures":   ROOT / "figures",      # exported charts
    "logs":      ROOT / "logs",         # run traces
}
for d in FOLDERS.values():
    d.mkdir(parents=True, exist_ok=True)

# --- 1.4  Report -------------------------------------------------------------
_free = shutil.disk_usage(ROOT).free / 1e9
SESSION_START = dt.datetime.now()

header("Runtime environment")
field("Platform detected", PLATFORM, SYM["ok"])
field("System", f"{platform.system()} {platform.release()} ({platform.machine()})", SYM["pt"])
field("Python", platform.python_version(), SYM["pt"])
field("Working root", str(ROOT), SYM["ok"])
for name, path in FOLDERS.items():
    field(f"  folder {name}", str(path.relative_to(ROOT)), SYM["pt"])
field("Free disk space", f"{_free:,.1f} GB",
      SYM["ok"] if _free > 5 else SYM["att"])
field("Timestamp", SESSION_START.strftime("%Y-%m-%d %H:%M:%S"), SYM["pt"])

if _free < 5:
    print(f"\n  {SYM['att']} Less than 5 GB available. Daily products are large:")
    print("     stay on VNP46A4 (annual), or lower MAX_FILES_PER_PRODUCT in step 4.")

<div style="background:#E8F5EF;border:1px solid #00704A44;border-radius:12px;padding:13px 18px;margin:10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><b style="color:#00704A;">ℹ️ Colab: keeping files between sessions</b><div style="color:#231F20;margin-top:5px;line-height:1.55;">The Colab machine is wiped when you close it. To keep the granules, mount your Google Drive <i>before</i> running the cell above and point the root at it:<br><code>from google.colab import drive; drive.mount('/content/drive')</code><br><code>import os; os.environ['NTL_HOME'] = '/content/drive/MyDrive/NTL_LOCAL'</code></div></div>

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">STEP 02 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Dependencies</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Two essential libraries, three optional ones</div></div>

The foundation is deliberately minimal: `requests` for the network and `pandas` for tables. Everything else is optional and never blocks execution — `h5py` opens the HDF5 files at step 9, `numpy` and `matplotlib` visualise them. No heavy geospatial library (`gdal`, `rasterio`, `geopandas`) is required: that is what makes this notebook installable in thirty seconds on any machine.

The cell installs **only what is missing**, which avoids breaking an already-configured local environment.

In [ ]:
# =============================================================================
#  STEP 2 - Dependencies: check, install what is missing, report
# =============================================================================
import importlib, importlib.util, subprocess

REQUIRED     = {"requests": "requests", "pandas": "pandas"}
OPTIONAL = {"h5py": "h5py", "numpy": "numpy", "matplotlib": "matplotlib"}


def install(packages):
    """Silent install through the running Python (works outside Jupyter too).

    Three successive attempts: plain install, then --user, then
    --break-system-packages. The last one is needed on recent Linux distributions
    (Debian, Ubuntu 23.04+) that refuse pip on the system Python (PEP 668).
    """
    base = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    for variant in ([], ["--user"], ["--break-system-packages"]):
        try:
            subprocess.check_call(base + variant + list(packages),
                                  stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
            return True
        except Exception:
            continue
    return False


def ensure(catalogue, blocking):
    """Install missing modules, then return {module: version | None}."""
    missing = [pkg for mod, pkg in catalogue.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print(f"  {SYM['fl']} Installing: {', '.join(missing)} ...")
        if not install(missing):
            msg = "install failed (no network access to PyPI?)"
            if blocking:
                raise RuntimeError(f"Required dependencies missing: {missing} - {msg}")
            print(f"  {SYM['att']} {msg} - carrying on without.")
        importlib.invalidate_caches()

    state = {}
    for mod in catalogue:
        try:
            m = importlib.import_module(mod)
            state[mod] = getattr(m, "__version__", "?")
        except Exception:
            state[mod] = None
    return state


header("Dependencies")
state_required = ensure(REQUIRED, blocking=True)
state_optional = ensure(OPTIONAL, blocking=False)

for mod, ver in state_required.items():
    field(f"{mod} (required)", ver, SYM["ok"])
for mod, ver in state_optional.items():
    field(f"{mod} (optional)", ver or "missing",
          SYM["ok"] if ver else SYM["att"])

import requests
import pandas as pd

# Flags used further down to cleanly disable what is not installed
H5PY_OK = state_optional.get("h5py") is not None
MPL_OK  = state_optional.get("matplotlib") is not None and state_optional.get("numpy") is not None

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">STEP 03 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">NASA Earthdata credentials</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">One token, four ways to supply it — never written in the notebook</div></div>

Black Marble files are **free**, but downloading them requires a NASA Earthdata account. Authentication uses a **bearer token**, not a password: a long string sent in the HTTP header `Authorization: Bearer <token>`.

**Getting a token, once:**

1. Create a free account at <https://urs.earthdata.nasa.gov>.
2. Open <https://urs.earthdata.nasa.gov/profile> → **Generate Token** tab → copy the string.
3. Hand it to the notebook through one of the four channels below.

| Environment | Recommended method |
|---|---|
| **Local Jupyter** | a `.env` file next to the notebook, containing `EARTHDATA_TOKEN=your_token` |
| **Google Colab** | 🔑 *Secrets* panel → new secret named `EARTHDATA_TOKEN`, access enabled for this notebook |
| **Kaggle** | *Add-ons → Secrets* → secret named `EARTHDATA_TOKEN` |
| **Anywhere** | masked interactive input, offered as a last resort (the token is not stored) |


<div style="background:#FFF6E0;border:1px solid #D49A0044;border-radius:12px;padding:13px 18px;margin:10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><b style="color:#D49A00;">⚠️ Never paste your token into a cell</b><div style="color:#231F20;margin-top:5px;line-height:1.55;">A token written in clear text travels with the notebook: into the workshop's GitHub repository, into an email attachment, into the version history. It grants access to your NASA account. The four channels above keep it <b>outside</b> the <code>.ipynb</code> file. If a token leaks, revoke it from the Earthdata profile page — that is immediate and free. Remember to add <code>.env</code> to your <code>.gitignore</code> too.</div></div>

In [ ]:
# =============================================================================
#  STEP 3 - Fetch the Earthdata token (never written in this notebook)
# =============================================================================
import getpass

# Set to False to never show an input prompt (unattended runs)
PROMPT_IF_MISSING = True

# Accepted names, by order of preference - compatible with the 2024 notebooks
TOKEN_NAMES = ["EARTHDATA_TOKEN", "EARTHDATA_BEARER", "LAADS_TOKEN"]


def read_dotenv(start=None, levels=3):
    """Read a .env in the current folder, then in its parents. No dependency."""
    start = Path(start or Path.cwd())
    candidates = [start / ".env"] + [p / ".env" for p in list(start.parents)[:levels]]
    candidates.append(Path.home() / ".env")
    values = {}
    for f in candidates:
        try:
            if not f.exists():
                continue
            for raw in f.read_text(encoding="utf-8").splitlines():
                raw = raw.strip()
                if not raw or raw.startswith("#") or "=" not in raw:
                    continue
                key, val = raw.split("=", 1)
                values.setdefault(key.strip(), val.strip().strip('"').strip("'"))
            if values:
                return values, f
        except Exception:
            continue
    return {}, None


def _from_colab():
    try:
        from google.colab import userdata          # type: ignore
        for n in TOKEN_NAMES:
            try:
                v = userdata.get(n)
                if v:
                    return v.strip()
            except Exception:
                continue
    except Exception:
        pass
    return None


def _from_kaggle():
    try:
        from kaggle_secrets import UserSecretsClient   # type: ignore
        client = UserSecretsClient()
        for n in TOKEN_NAMES:
            try:
                v = client.get_secret(n)
                if v:
                    return v.strip()
            except Exception:
                continue
    except Exception:
        pass
    return None


def get_token():
    """Walk the sources in order and return (token, origin)."""
    # 1) process environment variables
    for n in TOKEN_NAMES:
        v = os.environ.get(n, "").strip()
        if v:
            return v, f"environment variable {n}"

    # 2) .env file
    env, filename = read_dotenv()
    for n in TOKEN_NAMES:
        if env.get(n):
            return env[n].strip(), f"file {filename}"

    # 3) platform secret store
    if PLATFORM == "Colab":
        v = _from_colab()
        if v:
            return v, "Google Colab Secrets"
    if PLATFORM == "Kaggle":
        v = _from_kaggle()
        if v:
            return v, "Kaggle Secrets"

    # 4) masked interactive prompt
    if PROMPT_IF_MISSING:
        try:
            v = getpass.getpass("Earthdata token (hidden input, Enter to skip): ").strip()
            if v:
                return v, "interactive input (not stored)"
        except Exception:
            pass
    return "", "none"


TOKEN, TOKEN_SOURCE = get_token()

header("Earthdata credentials")
if TOKEN:
    fingerprint = f"{TOKEN[:6]}…{TOKEN[-4:]}" if len(TOKEN) > 14 else "(too short?)"
    field("Token detected", f"{len(TOKEN)} characters - {fingerprint}", SYM["ok"])
    field("Source", TOKEN_SOURCE, SYM["pt"])
    if len(TOKEN) < 40 or " " in TOKEN:
        print(f"\n  {SYM['att']} This token looks unusual (too short, or contains a space).")
        print("     Check that it was copied in full from urs.earthdata.nasa.gov/profile.")
else:
    field("Token detected", "none", SYM["att"])
    print(f"\n  {SYM['att']} Steps 1 to 7 still work: searching the NASA CMR catalogue")
    print("     is public. Only step 8 (download) needs the token.")

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">STEP 04 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Collection parameters</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">The only cell you need to edit</div></div>

Everything the notebook does is controlled by the cell below, and the principle behind the temporal grid is worth understanding before you change it.

### The principle: one fixed date, several years

Comparing 15 January 2015 with 3 August 2023 makes no sense for night-time lights. Radiance varies strongly with the season — vegetation, clouds, solar angle, snow at high latitudes — and a good share of the difference you would measure would be seasonality, not development. The remedy is simple and standard: fix the date, and vary only the year.

You therefore fill in **three things** instead of four windows:

| Parameter | What it fixes | Products concerned |
|---|---|---|
| `YEARS` | the years being compared | all four |
| `FIXED_DAY` | the day, in `MM-DD` format | `VNP46A1`, `VNP46A2` (daily) |
| `FIXED_MONTH` | the month, 1 to 12 | `VNP46A3` (monthly) |

With the default values — `FIXED_DAY = "01-15"`, `FIXED_MONTH = 1`, years 2013 to 2025 — the notebook fetches 15 January of thirteen consecutive years for the daily products, the thirteen months of January for the monthly one, and the thirteen annual composites. You get four comparable series, all aligned on the same season.

### Choosing your reference date

A good `FIXED_DAY` avoids both new and full Moon, and a cloudy season. In the tropics, the **dry season** gives the clearest nights: January for West Africa and the Sahel, July for southern Africa. A single day remains exposed to cloud: if the series has gaps, step 7 will show them year by year, and shifting the date by a few days is usually enough.

29 February is handled: non-leap years are simply flagged and skipped.

In [ ]:
# =============================================================================
#  STEP 4 - PARAMETERS  <-- the only cell you need to edit
# =============================================================================

# --- 4.1  The country --------------------------------------------------------
COUNTRY_ISO3 = "CIV"          # CIV, SDN, RWA, TUN, MAR, KEN, NGA, ZAF, EGY, SEN...

# Custom extent, if you want a finer area than a whole country:
# (west, south, east, north) in decimal degrees. Leave None to use the
# national extent from the built-in directory (step 5).
CUSTOM_BBOX = None       # e.g. (-4.30, 5.20, -3.70, 5.60) -> Greater Abidjan

BBOX_MARGIN_DEG = 0.0       # margin added around the extent, in degrees


# --- 4.2  The temporal grid: one fixed date, several years -------------------
YEARS    = list(range(2013, 2026))   # the years being compared
FIXED_DAY = "01-15"                   # MM-DD  -> for VNP46A1 and VNP46A2
FIXED_MONTH = 1                         # 1..12  -> for VNP46A3
                                      # VNP46A4 is annual: nothing to fix

PRODUCTS_TO_TEST = ["VNP46A1", "VNP46A2", "VNP46A3", "VNP46A4"]


# --- 4.3  Downloading --------------------------------------------------------
DOWNLOAD_MODE = "sample"           # "none" | "sample" | "full"
PRODUCTS_TO_DOWNLOAD = ["VNP46A4"]     # a list: add as many as you like
MAX_FILES_PER_PRODUCT = 2             # cap in "sample" mode


# --- 4.4  Product fact sheet (documentation, not a setting) ------------------
PRODUCT_CATALOGUE = {
    "VNP46A1": dict(label="Raw at-sensor radiance", frequency="daily",
                    since="2012-01-19", variable="DNB_At_Sensor_Radiance_500m",
                    doi="10.5067/VIIRS/VNP46A1.001",
                    usage="Diagnostics: see the data before any correction."),
    "VNP46A2": dict(label="Lunar BRDF-corrected, gap-filled NTL", frequency="daily",
                    since="2012-01-19", variable="Gap_Filled_DNB_BRDF-Corrected_NTL",
                    doi="10.5067/VIIRS/VNP46A2.001",
                    usage="Fine monitoring: crises, outages, dated events."),
    "VNP46A3": dict(label="Monthly composite", frequency="monthly",
                    since="2012-01-01", variable="NearNadir_Composite_Snow_Free",
                    doi="10.5067/VIIRS/VNP46A3.001",
                    usage="Seasonality and sub-annual series."),
    "VNP46A4": dict(label="Annual composite", frequency="annual",
                    since="2012-01-01", variable="NearNadir_Composite_Snow_Free",
                    doi="10.5067/VIIRS/VNP46A4.001",
                    usage="Long series, regional comparisons, indicators."),
}

MONTH_NAME = ["", "January", "February", "March", "April", "May", "June", "July",
              "August", "September", "October", "November", "December"]

# --- 4.5  Immediate consistency checks ---------------------------------------
assert DOWNLOAD_MODE in {"none", "sample", "full"}, \
    "DOWNLOAD_MODE must be 'none', 'sample' or 'full'."
assert set(PRODUCTS_TO_TEST) <= set(PRODUCT_CATALOGUE), \
    f"Unknown products: {set(PRODUCTS_TO_TEST) - set(PRODUCT_CATALOGUE)}"
assert set(PRODUCTS_TO_DOWNLOAD) <= set(PRODUCTS_TO_TEST) or DOWNLOAD_MODE == "none", \
    "Every product in PRODUCTS_TO_DOWNLOAD must appear in PRODUCTS_TO_TEST."
assert YEARS, "YEARS cannot be empty."
assert min(YEARS) >= 2012, "Black Marble starts in 2012: no data before that."
assert 1 <= FIXED_MONTH <= 12, "FIXED_MONTH must be between 1 and 12."
try:
    _m, _j = (int(x) for x in FIXED_DAY.split("-"))
    dt.date(2016, _m, _j)                     # 2016 is a leap year: validates 29 Feb
except Exception:
    raise AssertionError("FIXED_DAY must use the \"MM-DD\" format, for example \"01-15\".")

header("Parameters in use")
field("Country (ISO3)", COUNTRY_ISO3, SYM["ok"])
field("Extent", "custom" if CUSTOM_BBOX else "national (built-in directory)", SYM["pt"])
field("Years compared", f"{len(YEARS)} - from {min(YEARS)} to {max(YEARS)}", SYM["ok"])
field("Fixed day (daily)", f"{int(FIXED_DAY.split('-')[1])} {MONTH_NAME[int(FIXED_DAY.split('-')[0])]}",
      SYM["pt"])
field("Fixed month (monthly)", MONTH_NAME[FIXED_MONTH], SYM["pt"])
field("Products tested", ", ".join(PRODUCTS_TO_TEST), SYM["ok"])
field("Download mode", DOWNLOAD_MODE, SYM["ok"])
if DOWNLOAD_MODE != "none":
    field("  products targeted", ", ".join(PRODUCTS_TO_DOWNLOAD), SYM["pt"])
    field("  cap per product",
          "none (full)" if DOWNLOAD_MODE == "full"
          else f"{MAX_FILES_PER_PRODUCT} file(s)", SYM["pt"])

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">STEP 05 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">National extent and VIIRS tiles</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">From ISO3 code to tile list, by calculation</div></div>

Two operations follow one another here, and neither needs the network.

**a) From ISO3 code to an extent.** The notebook ships a directory of 182 national extents (bounding boxes derived from Natural Earth, including the 55 African Union member states). No external file, no network call: the notebook stays usable offline and inside a locked-down environment. If your country is missing, or if you are targeting an area finer than a country, set `CUSTOM_BBOX` in step 4.

**b) From extent to tiles.** The Black Marble grid is regular, so the conversion is a plain integer division — no server needed to know which tiles cover a country:

$$h = \left\lfloor \frac{\lambda + 180}{10} \right\rfloor \qquad v = \left\lfloor \frac{90 - \varphi}{10} \right\rfloor$$

where $\lambda$ is longitude and $\varphi$ latitude. The formula is applied to the four corners of the extent, and the Cartesian product of the resulting indices gives the tile set. The cell checks that calculation against two known cases — Côte d'Ivoire (`h17v07`, `h17v08`) and Sudan (six tiles) — before applying it to your country.

In [ ]:
# =============================================================================
#  STEP 5 - Country extent, then the VIIRS tiles that cover it
# =============================================================================
import math, difflib

# --- 5.1  Built-in directory: ISO3|ISO2|name|west|south|east|north|region ----
# National bounding boxes (decimal degrees, WGS84). The 55 African Union
# member states come first, the rest of the world after.
_BBOX_TABLE = """
AGO|AO|Angola|11.64|-17.93|24.08|-4.44|AF
BFA|BF|Burkina Faso|-5.47|9.61|2.18|15.12|AF
BDI|BI|Burundi|29.02|-4.50|30.75|-2.35|AF
BEN|BJ|Benin|0.77|6.14|3.80|12.24|AF
BWA|BW|Botswana|19.90|-26.83|29.43|-17.66|AF
COD|CD|DR Congo|12.18|-13.26|31.17|5.26|AF
CAF|CF|Central African Republic|14.46|2.27|27.37|11.14|AF
COG|CG|Congo-Brazzaville|11.09|-5.04|18.45|3.73|AF
CIV|CI|Côte d'Ivoire|-8.60|4.34|-2.56|10.52|AF
CMR|CM|Cameroon|8.49|1.73|16.01|12.86|AF
CPV|CV|Cabo Verde|-25.36|14.80|-22.66|17.21|AF
DJI|DJ|Djibouti|41.66|10.93|43.32|12.70|AF
DZA|DZ|Algeria|-8.68|19.06|12.00|37.12|AF
EGY|EG|Egypt|24.70|22.00|36.87|31.59|AF
ESH|EH|Western Sahara|-17.10|20.77|-8.67|27.67|AF
ERI|ER|Eritrea|36.32|12.46|43.08|18.00|AF
ETH|ET|Ethiopia|32.95|3.42|47.79|14.96|AF
GAB|GA|Gabon|8.80|-3.98|14.43|2.33|AF
GHA|GH|Ghana|-3.24|4.71|1.06|11.10|AF
GMB|GM|Gambia|-16.84|13.13|-13.84|13.88|AF
GIN|GN|Guinea|-15.13|7.31|-7.83|12.59|AF
GNQ|GQ|Equatorial Guinea|9.31|1.01|11.29|2.28|AF
GNB|GW|Guinea-Bissau|-16.68|11.04|-13.70|12.63|AF
KEN|KE|Kenya|33.89|-4.68|41.86|5.51|AF
COM|KM|Comoros|43.21|-12.42|44.54|-11.36|AF
LBR|LR|Liberia|-11.44|4.36|-7.54|8.54|AF
LSO|LS|Lesotho|27.00|-30.65|29.33|-28.65|AF
LBY|LY|Libya|9.32|19.58|25.16|33.14|AF
MAR|MA|Morocco|-17.02|21.42|-1.12|35.76|AF
MDG|MG|Madagascar|43.25|-25.60|50.48|-12.04|AF
MLI|ML|Mali|-12.17|10.10|4.27|24.97|AF
MRT|MR|Mauritania|-17.06|14.62|-4.92|27.40|AF
MUS|MU|Mauritius|57.30|-20.55|63.51|-19.65|AF
MWI|MW|Malawi|32.69|-16.80|35.77|-9.23|AF
MOZ|MZ|Mozambique|30.18|-26.74|40.78|-10.32|AF
NAM|NA|Namibia|11.73|-29.05|25.08|-16.94|AF
NER|NE|Niger|0.30|11.66|15.90|23.47|AF
NGA|NG|Nigeria|2.69|4.24|14.58|13.87|AF
RWA|RW|Rwanda|29.02|-2.92|30.82|-1.13|AF
SYC|SC|Seychelles|55.17|-4.85|55.95|-3.69|AF
SDN|SD|Sudan|21.94|8.62|38.41|22.00|AF
SLE|SL|Sierra Leone|-13.25|6.79|-10.23|10.05|AF
SEN|SN|Senegal|-17.63|12.33|-11.47|16.60|AF
SOM|SO|Somalia|40.98|-1.68|51.13|12.02|AF
SSD|SS|South Sudan|23.89|3.51|35.30|12.25|AF
STP|ST|Sao Tome and Principe|6.46|0.02|7.47|1.72|AF
SWZ|SZ|Eswatini|30.68|-27.29|32.07|-25.66|AF
TCD|TD|Chad|13.54|7.42|23.89|23.41|AF
TGO|TG|Togo|-0.05|5.93|1.87|11.02|AF
TUN|TN|Tunisia|7.52|30.31|11.49|37.35|AF
TZA|TZ|Tanzania|29.34|-11.72|40.32|-0.95|AF
UGA|UG|Uganda|29.58|-1.44|35.04|4.25|AF
ZAF|ZA|South Africa|16.34|-34.82|32.83|-22.09|AF
ZMB|ZM|Zambia|21.89|-17.96|33.49|-8.24|AF
ZWE|ZW|Zimbabwe|25.26|-22.27|32.85|-15.51|AF
ARE|AE|United Arab Emirates|51.58|22.50|56.40|26.06|--
AFG|AF|Afghanistan|60.53|29.32|75.16|38.49|--
ALB|AL|Albania|19.30|39.62|21.02|42.69|--
ARM|AM|Armenia|43.58|38.74|46.51|41.25|--
ATA|AQ|Antarctica|-180.00|-90.00|180.00|-63.27|--
ARG|AR|Argentina|-73.42|-55.25|-53.63|-21.83|--
AUT|AT|Austria|9.48|46.43|16.98|49.04|--
AUS|AU|Australia|113.34|-43.63|153.57|-10.67|--
AZE|AZ|Azerbaijan|44.79|38.27|50.39|41.86|--
BIH|BA|Bosnia and Herzegovina|15.75|42.65|19.60|45.23|--
BGD|BD|Bangladesh|88.08|20.67|92.67|26.45|--
BEL|BE|Belgium|2.51|49.53|6.16|51.48|--
BGR|BG|Bulgaria|22.38|41.23|28.56|44.23|--
BHR|BH|Bahrain|50.45|25.79|50.82|26.29|--
BRN|BN|Brunei Darussalam|114.20|4.01|115.45|5.45|--
BOL|BO|Bolivia|-69.59|-22.87|-57.50|-9.76|--
BRA|BR|Brazil|-73.99|-33.77|-34.73|5.24|--
BHS|BS|Bahamas|-78.98|23.71|-77.00|27.04|--
BTN|BT|Bhutan|88.81|26.72|92.10|28.30|--
BLR|BY|Belarus|23.20|51.32|32.69|56.17|--
BLZ|BZ|Belize|-89.23|15.89|-88.11|18.50|--
CAN|CA|Canada|-141.00|41.68|-52.65|73.23|--
CHE|CH|Switzerland|6.02|45.78|10.44|47.83|--
CHL|CL|Chile|-75.64|-55.61|-66.96|-17.58|--
CHN|CN|China|73.68|18.20|135.03|53.46|--
COL|CO|Colombia|-78.99|-4.30|-66.88|12.44|--
CRI|CR|Costa Rica|-85.94|8.23|-82.55|11.22|--
CUB|CU|Cuba|-84.97|19.86|-74.18|23.19|--
CYP|CY|Cyprus|32.26|34.57|34.00|35.17|--
CZE|CZ|Czechia|12.24|48.56|18.85|51.12|--
DEU|DE|Germany|5.99|47.30|15.02|54.98|--
DNK|DK|Denmark|8.09|54.80|12.69|57.73|--
DOM|DO|Dominican Republic|-71.95|17.60|-68.32|19.88|--
ECU|EC|Ecuador|-80.97|-4.96|-75.23|1.38|--
EST|EE|Estonia|23.34|57.47|28.13|59.61|--
ESP|ES|Spain|-9.39|35.95|3.04|43.75|--
FIN|FI|Finland|20.65|59.85|31.52|70.16|--
FJI|FJ|Fiji|-180.00|-18.29|180.00|-16.02|--
FLK|FK|Falkland Islands (Malvinas)|-61.20|-52.30|-57.75|-51.10|--
FRA|FR|France|-5.00|42.50|9.56|51.15|--
GBR|GB|United Kingdom|-7.57|49.96|1.68|58.64|--
GEO|GE|Georgia|39.96|41.06|46.64|43.55|--
GRL|GL|Greenland|-73.30|60.04|-12.21|83.65|--
GRC|GR|Greece|20.15|34.92|26.60|41.83|--
GTM|GT|Guatemala|-92.23|13.74|-88.23|17.82|--
GUY|GY|Guyana|-61.41|1.27|-56.54|8.37|--
HND|HN|Honduras|-89.35|12.98|-83.15|16.01|--
HRV|HR|Croatia|13.66|42.48|19.39|46.50|--
HTI|HT|Haiti|-74.46|18.03|-71.62|19.92|--
HUN|HU|Hungary|16.20|45.76|22.71|48.62|--
IDN|ID|Indonesia|95.29|-10.36|141.03|5.48|--
IRL|IE|Ireland|-9.98|51.67|-6.03|55.13|--
ISR|IL|Israel|34.27|29.50|35.84|33.28|--
IND|IN|India|68.18|7.97|97.40|35.49|--
IRQ|IQ|Iraq|38.79|29.10|48.57|37.39|--
IRN|IR|Iran|44.11|25.08|63.32|39.71|--
ISL|IS|Iceland|-24.33|63.50|-13.61|66.53|--
ITA|IT|Italy|6.75|36.62|18.48|47.12|--
JAM|JM|Jamaica|-78.34|17.70|-76.20|18.52|--
JOR|JO|Jordan|34.92|29.20|39.20|33.38|--
JPN|JP|Japan|129.41|31.03|145.54|45.55|--
KGZ|KG|Kyrgyzstan|69.46|39.28|80.26|43.30|--
KHM|KH|Cambodia|102.35|10.49|107.61|14.57|--
PRK|KP|North Korea|124.27|37.67|130.78|42.99|--
KOR|KR|South Korea|126.12|34.39|129.47|38.61|--
KWT|KW|Kuwait|46.57|28.53|48.42|30.06|--
KAZ|KZ|Kazakhstan|46.47|40.66|87.36|55.39|--
LAO|LA|Laos|100.12|13.88|107.56|22.46|--
LBN|LB|Lebanon|35.13|33.09|36.61|34.64|--
LKA|LK|Sri Lanka|79.70|5.97|81.79|9.82|--
LTU|LT|Lithuania|21.06|53.91|26.59|56.37|--
LUX|LU|Luxembourg|5.67|49.44|6.24|50.13|--
LVA|LV|Latvia|21.06|55.62|28.18|57.97|--
MDA|MD|Moldova|26.62|45.49|30.02|48.47|--
MNE|ME|Montenegro|18.45|41.88|20.34|43.52|--
MKD|MK|North Macedonia|20.46|40.84|22.95|42.32|--
MMR|MM|Myanmar|92.30|9.93|101.18|28.34|--
MNG|MN|Mongolia|87.75|41.60|119.77|52.05|--
MLT|MT|Malta|14.18|35.79|14.58|36.08|--
MEX|MX|Mexico|-117.13|14.54|-86.81|32.72|--
MYS|MY|Malaysia|100.09|0.77|119.18|6.93|--
NCL|NC|New Caledonia|164.03|-22.40|167.12|-20.11|--
NIC|NI|Nicaragua|-87.67|10.73|-83.15|15.02|--
NLD|NL|Netherlands|3.31|50.80|7.09|53.51|--
NOR|NO|Norway|4.99|58.08|31.29|70.92|--
NPL|NP|Nepal|80.09|26.40|88.17|30.42|--
NZL|NZ|New Zealand|166.51|-46.64|178.52|-34.45|--
OMN|OM|Oman|52.00|16.65|59.81|26.40|--
PAN|PA|Panama|-82.97|7.22|-77.24|9.61|--
PER|PE|Peru|-81.41|-18.35|-68.67|-0.06|--
PNG|PG|Papua New Guinea|141.00|-10.65|156.02|-2.50|--
PHL|PH|Philippines|117.17|5.58|126.54|18.51|--
PAK|PK|Pakistan|60.87|23.69|77.84|37.13|--
POL|PL|Poland|14.07|49.03|24.03|54.85|--
PRI|PR|Puerto Rico|-67.24|17.95|-65.59|18.52|--
PSE|PS|Palestine|34.93|31.35|35.55|32.53|--
PRT|PT|Portugal|-9.53|36.84|-6.39|42.28|--
PRY|PY|Paraguay|-62.69|-27.55|-54.29|-19.34|--
QAT|QA|Qatar|50.74|24.56|51.61|26.11|--
ROU|RO|Romania|20.22|43.69|29.63|48.22|--
SRB|RS|Serbia|18.83|42.25|22.99|46.17|--
RUS|RU|Russia|-180.00|41.15|180.00|81.25|--
SAU|SA|Saudi Arabia|34.63|16.35|55.67|32.16|--
SLB|SB|Solomon Islands|156.49|-10.83|162.40|-6.60|--
SWE|SE|Sweden|11.03|55.36|23.90|69.11|--
SGP|SG|Singapore|103.60|1.15|104.09|1.47|--
SVN|SI|Slovenia|13.70|45.45|16.56|46.85|--
SVK|SK|Slovakia|16.88|47.76|22.56|49.57|--
SUR|SR|Suriname|-58.04|1.82|-53.96|6.03|--
SLV|SV|El Salvador|-90.10|13.15|-87.72|14.42|--
SYR|SY|Syria|35.70|32.31|42.35|37.23|--
ATF|TF|French Southern Territories|68.72|-49.78|70.56|-48.63|--
THA|TH|Thailand|97.38|5.69|105.59|20.42|--
TJK|TJ|Tajikistan|67.44|36.74|74.98|40.96|--
TLS|TL|Timor-Leste|124.97|-9.39|127.34|-8.27|--
TKM|TM|Turkmenistan|52.50|35.27|66.55|42.75|--
TUR|TR|Türkiye|26.04|35.82|44.79|42.14|--
TTO|TT|Trinidad and Tobago|-61.95|10.00|-60.90|10.89|--
TWN|TW|Taiwan|120.11|21.97|121.95|25.30|--
UKR|UA|Ukraine|22.09|44.36|40.08|52.34|--
USA|US|United States|-125.00|25.00|-66.96|49.50|--
URY|UY|Uruguay|-58.43|-34.95|-53.21|-30.11|--
UZB|UZ|Uzbekistan|55.93|37.14|73.06|45.59|--
VEN|VE|Venezuela|-73.30|0.72|-59.76|12.16|--
VNM|VN|Vietnam|102.17|8.60|109.34|23.35|--
VUT|VU|Vanuatu|166.63|-16.60|167.84|-14.63|--
YEM|YE|Yemen|42.60|12.59|53.11|19.00|--
"""


def _load_bboxes():
    table = {}
    for raw in _BBOX_TABLE.strip().splitlines():
        iso3, iso2, name, o, s, e, n, reg = raw.split("|")
        table[iso3] = dict(iso2=iso2, name=name, region=reg,
                           bbox=(float(o), float(s), float(e), float(n)))
    return table


BBOXES = _load_bboxes()


def resolve_country(iso3, custom_bbox=None, margin=0.0):
    """Return (name, bbox) for an ISO3 code. Explicit help message if unknown."""
    iso3 = (iso3 or "").strip().upper()

    if iso3 not in BBOXES and custom_bbox is None:
        close = difflib.get_close_matches(iso3, BBOXES.keys(), n=4, cutoff=0.5)
        hint = f" Close codes: {', '.join(close)}." if close else ""
        raise KeyError(
            f"ISO3 code '{iso3}' is not in the directory.{hint}\n"
            "    -> Use an alpha-3 code (CIV, SDN, RWA...), or set "
            "CUSTOM_BBOX = (west, south, east, north) in step 4."
        )

    name = BBOXES[iso3]["name"] if iso3 in BBOXES else f"custom area ({iso3})"
    o, s, e, n = custom_bbox if custom_bbox else BBOXES[iso3]["bbox"]

    if margin:
        o, s, e, n = o - margin, s - margin, e + margin, n + margin
    # Physical bounds of the globe
    o, e = max(-180.0, o), min(180.0, e)
    s, n = max(-90.0, s), min(90.0, n)
    if not (o < e and s < n):
        raise ValueError(f"Inconsistent extent: {(o, s, e, n)} - expected (west, south, east, north).")
    return name, (round(o, 4), round(s, 4), round(e, 4), round(n, 4))


def tiles_from_bbox(bbox):
    """Apply h = (lon+180)//10 and v = (90-lat)//10 to the corners of the extent."""
    o, s, e, n = bbox
    eps = 1e-9                       # avoids a spurious tile when an edge falls exactly on 10 deg
    h0 = int(math.floor((o + 180.0) / 10.0))
    h1 = int(math.floor((e + 180.0 - eps) / 10.0))
    v0 = int(math.floor((90.0 - n) / 10.0))
    v1 = int(math.floor((90.0 - s - eps) / 10.0))
    h0, h1 = max(0, min(h0, 35)), max(0, min(h1, 35))
    v0, v1 = max(0, min(v0, 17)), max(0, min(v1, 17))
    return [f"h{h:02d}v{v:02d}" for v in range(v0, v1 + 1) for h in range(h0, h1 + 1)]


# --- 5.2  Self-test: the formula must reproduce known tile sets --------------
_expected = {"CIV": {"h17v07", "h17v08"},
             "SDN": {"h20v06", "h21v06", "h20v07", "h21v07", "h20v08", "h21v08"}}
for _iso, _ref in _expected.items():
    _got = set(tiles_from_bbox(resolve_country(_iso)[1]))
    assert _got == _ref, f"Tile self-test failed for {_iso}: {_got} != {_ref}"

# --- 5.3  Applied to the requested country -----------------------------------
COUNTRY_NAME, BBOX = resolve_country(COUNTRY_ISO3, CUSTOM_BBOX, BBOX_MARGIN_DEG)
TILES = tiles_from_bbox(BBOX)

header(f"Study area - {COUNTRY_NAME}")
field("ISO3 code", COUNTRY_ISO3, SYM["ok"])
field("Extent (W, S, E, N)", ", ".join(f"{v:+.2f}°" for v in BBOX), SYM["pt"])
field("Span", f"{BBOX[2]-BBOX[0]:.2f} deg x {BBOX[3]-BBOX[1]:.2f} deg", SYM["pt"])
field("VIIRS tiles", f"{len(TILES)} - {', '.join(TILES)}", SYM["ok"])
print(f"\n  {SYM['pt']} Tile formula self-test: Cote d'Ivoire and Sudan {SYM['ok']}")
print(f"  {SYM['pt']} One granule = 1 tile x 1 date. Files = periods x {len(TILES)} tiles.")

### The tile fact sheet

The table produced below is the **geographic reference** for your collection. For each selected tile it gives the identifier, the four corners in degrees, the share of the national extent it carries, and — most useful for what comes next — **the pixel window** to crop from the 2,400 × 2,400 matrix so that only your country remains.

| Column | What it holds |
|---|---|
| `tile`, `h`, `v` | the identifier and its two grid indices |
| `lon_min` … `lat_max` | the four corners of the tile, in decimal degrees |
| `share_of_bbox_pct` | the share of the national extent this tile carries (the shares add up to 100 %) |
| `col_min`, `col_max`, `row_min`, `row_max` | the pixel window to crop, out of 2,400 × 2,400 |
| `example_granule` | the exact name a file of this tile will have |

Each tile covers 10° × 10° in 2,400 × 2,400 pixels, that is **15 arc-seconds** per pixel — about 460 m at the equator. Converting a coordinate into a pixel is direct: `column = (λ − lon_min) / (10/2400)` and `row = (lat_max − φ) / (10/2400)`, with row 0 at the very top of the tile.

In [ ]:
# =============================================================================
#  STEP 5b - Reference table of the selected tiles
# =============================================================================
PIXELS_PER_TILE = 2400                       # Black Marble grid: 2400 x 2400
DEGREES_PER_PIXEL = 10.0 / PIXELS_PER_TILE    # 15 arc-seconds (~460 m at the equator)


def tile_reference(bbox, tiles, sample_product="VNP46A4", sample_year=2024):
    # Fact sheet for each tile: extent, share of the country, pixel window.
    o, s, e, n = bbox
    bbox_area = (e - o) * (n - s)
    rows = []

    for t in tiles:
        h, v = int(t[1:3]), int(t[4:6])
        t_o, t_e = h * 10.0 - 180.0, (h + 1) * 10.0 - 180.0      # tile bounds
        t_n, t_s = 90.0 - v * 10.0, 90.0 - (v + 1) * 10.0

        # Intersection between the tile and the national extent
        i_o, i_e = max(o, t_o), min(e, t_e)
        i_s, i_n = max(s, t_s), min(n, t_n)
        share = 100.0 * max(0.0, i_e - i_o) * max(0.0, i_n - i_s) / bbox_area if bbox_area else 0.0

        # Pixel window to crop inside the 2400 x 2400 matrix
        col_min = max(0, int((i_o - t_o) / DEGREES_PER_PIXEL))
        col_max = min(PIXELS_PER_TILE, int(math.ceil((i_e - t_o) / DEGREES_PER_PIXEL)))
        row_min = max(0, int((t_n - i_n) / DEGREES_PER_PIXEL))
        row_max = min(PIXELS_PER_TILE, int(math.ceil((t_n - i_s) / DEGREES_PER_PIXEL)))

        # Relative position of the tile in the set (useful when there are several)
        vs = sorted({int(x[4:6]) for x in tiles})
        hs = sorted({int(x[1:3]) for x in tiles})
        ns = "north" if v == vs[0] else ("south" if v == vs[-1] else "centre")
        oe = "west" if h == hs[0] else ("east" if h == hs[-1] else "centre")
        position = ns if len(vs) > 1 else ""
        position = (position + "-" + oe).strip("-") if len(hs) > 1 else position

        rows.append(dict(
            tile=t, h=h, v=v,
            lon_min=round(t_o, 2), lon_max=round(t_e, 2),
            lat_min=round(t_s, 2), lat_max=round(t_n, 2),
            position=position or "single",
            share_of_bbox_pct=round(share, 1),
            col_min=col_min, col_max=col_max, row_min=row_min, row_max=row_max,
            useful_pixels=(col_max - col_min) * (row_max - row_min),
            example_granule=f"{sample_product}.A{sample_year}001.{t}.001.<timestamp>.h5",
        ))
    return pd.DataFrame(rows)


TILE_REFERENCE = tile_reference(BBOX, TILES)

header(f"Tile reference - {COUNTRY_NAME} ({COUNTRY_ISO3})")

for _, r in TILE_REFERENCE.iterrows():
    print(f"\n  {SYM['ok']} {r['tile']}   (h = {r['h']:02d}, v = {r['v']:02d})"
          f"   position: {r['position']}")
    print(f"       extent           longitude {r['lon_min']:+7.2f} {SYM['fl']} {r['lon_max']:+7.2f}"
          f"   |   latitude {r['lat_min']:+6.2f} {SYM['fl']} {r['lat_max']:+6.2f}")
    print(f"       share of the national extent carried by this tile: {r['share_of_bbox_pct']:.1f} %")
    print(f"       crop window          columns {r['col_min']:4d} {SYM['fl']} {r['col_max']:4d}"
          f"   |   rows {r['row_min']:4d} {SYM['fl']} {r['row_max']:4d}"
          f"   ({r['useful_pixels']/1e6:.2f} M pixels out of 5.76 M)")
    print(f"       file name            {r['example_granule']}")

print(f"\n  {SYM['pt']} Grid: {PIXELS_PER_TILE} x {PIXELS_PER_TILE} pixels per tile, "
      f"{DEGREES_PER_PIXEL*3600:.0f} arc-seconds per pixel (~460 m at the equator)")
print(f"  {SYM['pt']} Shares add up to: {TILE_REFERENCE['share_of_bbox_pct'].sum():.1f} % "
      f"(must be 100 % - the extent is fully covered)")

_tiles_file = FOLDERS["catalogue"] / f"tiles_{COUNTRY_ISO3}.csv"
TILE_REFERENCE.to_csv(_tiles_file, index=False, encoding="utf-8-sig")
print(f"\n  {SYM['ok']} Reference table written: {_tiles_file}")
print(TILE_REFERENCE[["tile", "lon_min", "lon_max", "lat_min", "lat_max",
                        "share_of_bbox_pct", "col_min", "col_max",
                        "row_min", "row_max"]].to_string(index=False))

The figure below places the country inside the Black Marble grid. It serves a real control purpose: if the gold rectangle does not surround the country you are targeting, the ISO3 code or the custom extent is wrong — better to see that now than after downloading the wrong tiles.

In [ ]:
# =============================================================================
#  STEP 5c - Visual check: the country inside the tile grid
# =============================================================================
if MPL_OK:
    import numpy as np
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle

    o, s, e, n = BBOX
    h_idx = sorted({int(t[1:3]) for t in TILES})
    v_idx = sorted({int(t[4:6]) for t in TILES})
    h_view = range(max(0, h_idx[0] - 1), min(35, h_idx[-1] + 1) + 1)
    v_view = range(max(0, v_idx[0] - 1), min(17, v_idx[-1] + 1) + 1)

    fig_w = min(11.0, max(5.4, 1.95 * len(list(h_view)) + 2.0))
    fig_h = min(9.0, max(4.2, 1.95 * len(list(v_view)) + 1.3))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=110)
    for v in v_view:
        for h in h_view:
            x, y = h * 10 - 180, 90 - (v + 1) * 10
            selected = f"h{h:02d}v{v:02d}" in TILES
            ax.add_patch(Rectangle((x, y), 10, 10,
                                   facecolor="#E8F5EF" if selected else "#FFFFFF",
                                   edgecolor="#00A86A" if selected else "#D5DED9",
                                   linewidth=1.6 if selected else 0.8, zorder=1))
            ax.text(x + 5, y + 5.6 if selected else y + 5, f"h{h:02d}v{v:02d}",
                    ha="center", va="center", fontsize=8.5,
                    color="#00704A" if selected else "#B6C2BC",
                    fontweight="bold" if selected else "normal", zorder=2)
            if selected:                       # share of the extent carried by the tile
                share = float(TILE_REFERENCE.loc[
                    TILE_REFERENCE["tile"] == f"h{h:02d}v{v:02d}", "share_of_bbox_pct"].iloc[0])
                ax.text(x + 5, y + 4.0, f"{share:.1f} %", ha="center",
                        va="center", fontsize=8.2, color="#5E6964", zorder=2)

    ax.add_patch(Rectangle((o, s), e - o, n - s, facecolor="#F5C24233",
                           edgecolor="#D49A00", linewidth=2.2, zorder=3))
    ax.set_xlim(min(h_view) * 10 - 180, (max(h_view) + 1) * 10 - 180)
    ax.set_ylim(90 - (max(v_view) + 1) * 10, 90 - min(v_view) * 10)
    ax.set_aspect("equal")
    ax.set_xlabel("Longitude (deg)", fontsize=9, color="#5E6964")
    ax.set_ylabel("Latitude (deg)", fontsize=9, color="#5E6964")
    ax.tick_params(labelsize=8, colors="#5E6964")
    for spine in ax.spines.values():
        spine.set_color("#D5DED9")
    ax.set_title(f"{COUNTRY_NAME} ({COUNTRY_ISO3}) - {len(TILES)} tile(s) of 10x10 deg\n"
                 f"the percentage is the share of the extent carried by each tile",
                 fontsize=11.5, fontweight="bold", color="#231F20", pad=12)
    fig.tight_layout()
    _fig = FOLDERS["figures"] / f"tiles_{COUNTRY_ISO3}.png"
    fig.savefig(_fig, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  {SYM['ok']} Figure saved: {_fig}")
else:
    print(f"  {SYM['att']} matplotlib missing - visual check skipped.")
    print(f"     Extent {BBOX}  {SYM['fl']}  tiles {TILES}")

### The same thing on a real map

The abstract grid above tells you *where the tiles are*. The interactive map below tells you *what is inside them*: it lays the same rectangles over a real geographic basemap, with borders, cities and terrain. It is the most telling check — you see at once whether the extent matches the intended country, and how far a 10° tile overshoots a small one.

**Click a tile** and its reference sheet opens — corners in degrees, share of the extent, pixel window, expected file name. The selector in the top right switches between a light basemap, to read borders, and a dark one, more natural for night-time lights.

The map is also saved as **standalone HTML** in `figures/`. That file opens in any browser, travels by email, and publishes as-is on GitHub Pages — which will come in handy on Friday.

In [ ]:
# =============================================================================
#  STEP 5d - Interactive map: the tiles over a geographic basemap
# =============================================================================
def ensure_folium():
    # folium is a Python wrapper around Leaflet: pure Python, with no compiled
    # geospatial dependency. Shipped by default on Colab and Kaggle.
    try:
        import folium
        return folium
    except ModuleNotFoundError:
        print(f"  {SYM['fl']} Installation de folium ...")
        if install(["folium"]):
            try:
                importlib.invalidate_caches()
                import folium
                return folium
            except Exception:
                pass
    return None


folium = ensure_folium()

if folium is None:
    print(f"  {SYM['att']} folium unavailable - the static grid above still stands.")
else:
    b_o, b_s, b_e, b_n = BBOX
    fmap = folium.Map(location=[(b_s + b_n) / 2, (b_o + b_e) / 2],
                       zoom_start=5, tiles=None, control_scale=True)
    folium.TileLayer('OpenStreetMap', name='Light basemap').add_to(fmap)
    folium.TileLayer('CartoDB dark_matter', name='Dark basemap').add_to(fmap)

    # --- The Black Marble tiles, one by one ----------------------------------
    group = folium.FeatureGroup(name=f'VIIRS tiles ({len(TILES)})', show=True)
    share_max = max(1e-9, float(TILE_REFERENCE['share_of_bbox_pct'].max()))

    for _, r in TILE_REFERENCE.iterrows():
        popup_html = (
            '<div style="font-family:Segoe UI,Calibri,sans-serif;font-size:12.5px;min-width:290px;">'
            '<div style="background:#00553A;color:#fff;padding:7px 11px;border-radius:7px 7px 0 0;'
            'font-weight:700;font-size:14px;">'
            f'{r["tile"]}'
            '<span style="color:#F5C242;font-weight:400;font-size:11.5px;">'
            f' &middot; h={r["h"]:02d} v={r["v"]:02d} &middot; {r["position"]}</span></div>'
            '<table style="border-collapse:collapse;width:100%;">'
            '<tr><td style="padding:4px 10px;color:#5E6964;">Longitude</td>'
            f'<td style="padding:4px 10px;"><b>{r["lon_min"]:+.2f}&deg; &rarr; {r["lon_max"]:+.2f}&deg;</b></td></tr>'
            '<tr style="background:#F4F7F5;"><td style="padding:4px 10px;color:#5E6964;">Latitude</td>'
            f'<td style="padding:4px 10px;"><b>{r["lat_min"]:+.2f}&deg; &rarr; {r["lat_max"]:+.2f}&deg;</b></td></tr>'
            '<tr><td style="padding:4px 10px;color:#5E6964;">Share of the extent</td>'
            f'<td style="padding:4px 10px;color:#00704A;"><b>{r["share_of_bbox_pct"]:.1f} %</b></td></tr>'
            '<tr style="background:#F4F7F5;"><td style="padding:4px 10px;color:#5E6964;">Columns to crop</td>'
            f'<td style="padding:4px 10px;"><b>{r["col_min"]} &rarr; {r["col_max"]}</b> / 2400</td></tr>'
            '<tr><td style="padding:4px 10px;color:#5E6964;">Rows to crop</td>'
            f'<td style="padding:4px 10px;"><b>{r["row_min"]} &rarr; {r["row_max"]}</b> / 2400</td></tr>'
            '<tr style="background:#F4F7F5;"><td style="padding:4px 10px;color:#5E6964;">Pixels utiles</td>'
            f'<td style="padding:4px 10px;"><b>{r["useful_pixels"]/1e6:.2f} M</b> of 5.76 M</td></tr>'
            '</table>'
            '<div style="padding:7px 10px;background:#E8F5EF;border-radius:0 0 7px 7px;'
            'font-family:Consolas,monospace;font-size:10.5px;word-break:break-all;">'
            f'{r["example_granule"]}</div></div>'
        )

        # Green intensity proportional to the share of the extent carried
        weight = float(r['share_of_bbox_pct']) / share_max
        folium.Rectangle(
            bounds=[[r['lat_min'], r['lon_min']], [r['lat_max'], r['lon_max']]],
            color='#00A86A', weight=2.2, fill=True, fill_color='#00A86A',
            fill_opacity=0.06 + 0.20 * weight,
            tooltip=f'{r["tile"]}: {r["share_of_bbox_pct"]:.1f} % of the extent',
            popup=folium.Popup(popup_html, max_width=340),
        ).add_to(group)

        label_html = (
            '<div style="font-family:Segoe UI,Calibri,sans-serif;text-align:center;'
            'white-space:nowrap;transform:translate(-50%,-50%);">'
            '<div style="color:#00553A;font-weight:800;font-size:15px;'
            'text-shadow:0 0 4px #fff,0 0 9px #fff;">'
            f'{r["tile"]}</div>'
            '<div style="color:#00704A;font-size:11.5px;'
            'text-shadow:0 0 4px #fff,0 0 9px #fff;">'
            f'{r["share_of_bbox_pct"]:.1f} %</div></div>'
        )
        folium.Marker(
            location=[(r['lat_min'] + r['lat_max']) / 2, (r['lon_min'] + r['lon_max']) / 2],
            icon=folium.DivIcon(html=label_html),
        ).add_to(group)
    group.add_to(fmap)

    # --- The national extent being queried -----------------------------------
    extent_layer = folium.FeatureGroup(name='Queried extent', show=True)
    folium.Rectangle(
        bounds=[[b_s, b_o], [b_n, b_e]], color='#D49A00', weight=3, dash_array='9,6',
        fill=True, fill_color='#F5C242', fill_opacity=0.10,
        tooltip=f'Queried extent: {COUNTRY_NAME} ({COUNTRY_ISO3})',
        popup=folium.Popup(
            f'<b>{COUNTRY_NAME} ({COUNTRY_ISO3})</b><br>'
            f'West {b_o:+.2f}&deg; &middot; South {b_s:+.2f}&deg;<br>'
            f'East {b_e:+.2f}&deg; &middot; North {b_n:+.2f}&deg;<br>'
            f'Span {b_e-b_o:.2f}&deg; &times; {b_n-b_s:.2f}&deg;', max_width=260),
    ).add_to(extent_layer)
    extent_layer.add_to(fmap)

    # --- Floating legend -----------------------------------------------------
    legend = (
        '<div style="position:fixed;bottom:22px;left:14px;z-index:9999;background:#fff;'
        'border:1px solid #D5DED9;border-radius:10px;padding:11px 15px;'
        'font-family:Segoe UI,Calibri,sans-serif;font-size:11.5px;color:#231F20;'
        'box-shadow:0 2px 10px #00000022;">'
        '<div style="color:#00704A;font-weight:800;letter-spacing:1.5px;font-size:10px;">'
        'NASA BLACK MARBLE &middot; VNP46</div>'
        '<div style="font-weight:700;margin:3px 0 7px 0;font-size:13px;">'
        f'{COUNTRY_NAME} ({COUNTRY_ISO3})</div>'
        '<div><span style="display:inline-block;width:13px;height:13px;background:#00A86A44;'
        'border:2px solid #00A86A;vertical-align:-2px;"></span>&nbsp;'
        f'{len(TILES)} tile(s) of 10&deg;&times;10&deg;</div>'
        '<div style="margin-top:4px;"><span style="display:inline-block;width:13px;height:13px;'
        'background:#F5C24233;border:2px dashed #D49A00;vertical-align:-2px;"></span>'
        '&nbsp;queried extent</div>'
        '<div style="color:#5E6964;margin-top:7px;font-size:10.5px;">'
        'Click a tile for its reference sheet</div></div>'
    )
    fmap.get_root().html.add_child(folium.Element(legend))
    folium.LayerControl(collapsed=False).add_to(fmap)
    fmap.fit_bounds([[b_s - 1, b_o - 1], [b_n + 1, b_e + 1]])

    _map_file = FOLDERS['figures'] / f'map_tiles_{COUNTRY_ISO3}.html'
    fmap.save(str(_map_file))
    print(f"  {SYM['ok']} Interactive map: {_map_file}")
    print(f"  {SYM['pt']} Standalone HTML: opens outside Jupyter, publishable on GitHub Pages")

    try:
        from IPython.display import display
        display(fmap)
    except Exception:
        print(f"  {SYM['att']} Inline display unavailable: open the HTML file.")

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">STEP 06 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Querying the NASA CMR catalogue</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">One narrow window per year, paginated and deduplicated</div></div>

The entry point is a single URL: `https://cmr.earthdata.nasa.gov/search/granules.json`. You pass it the product short name, a temporal window and an extent; it answers in JSON, with each granule's exact name, its size and its download links.

### One request per year, not one big request

The "fixed date × years" grid translates into **one narrow window per year**. For thirteen years of `VNP46A2`, the notebook sends thirteen requests each asking for a single day, rather than one request spanning thirteen years from which 99.98 % of the result would then be thrown away. It is faster, it does not needlessly load NASA's server, and above all it makes diagnosis possible: if a year is missing, you know immediately which one.

The window depends on the product's frequency:

| Product | Window requested for year Y | Label |
|---|---|---|
| `VNP46A1`, `VNP46A2` | the single day `Y-MM-DD` | `2019-01-15` |
| `VNP46A3` | from the 1st to the last day of the fixed month | `2019-01` |
| `VNP46A4` | the whole year | `2019` |

### Three precautions that make the difference

**Pagination.** CMR caps results at 2,000 per page. The windows are narrow here, but the loop is there for the day you widen them — without it, part of the catalogue would be lost silently.

**Deduplication.** The same granule can be published under several collection versions (`.001`, `.002`). We keep a single entry per *(tile, date)* pair, retaining the highest version.

**Link selection.** Among the returned links are OPeNDAP and S3 URLs, which do not download like an ordinary file. We filter to keep only the direct HTTPS link to the `.h5`.

In [ ]:
# =============================================================================
#  STEP 6 - Temporal windows and CMR querying (no request issued here)
# =============================================================================
import re, time, json

CMR_URL = "https://cmr.earthdata.nasa.gov/search/granules.json"
CLIENT_ID = "BAD-GTS17-NTL"          # courtesy: identifies the caller to CMR
PAGE_SIZE = 2000                   # maximum allowed by CMR
MAX_PAGES = 25                       # safety net: 50,000 granules at most

GRANULE_PATTERN = re.compile(r"(VNP46A\d)\.A(\d{4})(\d{3})\.(h\d{2}v\d{2})\.(\d{3})\.", re.IGNORECASE)

SESSION_CMR = requests.Session()
SESSION_CMR.headers.update({"Client-Id": CLIENT_ID, "User-Agent": f"{CLIENT_ID}/1.0"})


# --- 6.1  The temporal grid: fixed date x years ------------------------------
def window_for_year(product, year):
    """Window to request for this product and this year.

    Returns (label, start date, end date), or None when the date does not
    exist that year (29 February outside a leap year).
    """
    freq = PRODUCT_CATALOGUE[product]["frequency"]

    if freq == "daily":
        month, day = (int(x) for x in FIXED_DAY.split("-"))
        try:
            d = dt.date(year, month, day)
        except ValueError:
            return None                       # 29 February, non-leap year
        return (d.isoformat(), d, d)

    if freq == "monthly":
        d0 = dt.date(year, FIXED_MONTH, 1)
        d1 = (dt.date(year + (FIXED_MONTH == 12), FIXED_MONTH % 12 + 1, 1)
              - dt.timedelta(days=1))
        return (f"{year}-{FIXED_MONTH:02d}", d0, d1)

    return (str(year), dt.date(year, 1, 1), dt.date(year, 12, 31))


def windows(product, years):
    """All valid windows of a product, in chronological order."""
    return [f for f in (window_for_year(product, a) for a in years) if f]


# --- 6.2  Reading a CMR response ---------------------------------------------
def _parse_name(filename):
    """VNP46A4.A2023001.h17v08.001.2024031102344.h5 → dict ou None."""
    m = GRANULE_PATTERN.search(filename or "")
    if not m:
        return None
    product, year, day, tile, version = m.groups()
    date = dt.date(int(year), 1, 1) + dt.timedelta(days=int(day) - 1)
    return dict(product=product.upper(), year=int(year), doy=int(day),
                tile=tile.lower(), version=int(version), date=date)


def _pick_url(links, filename):
    """Keep the direct HTTPS link to the .h5; drop OPeNDAP, S3 and thumbnails."""
    candidates = []
    for link in links or []:
        href = (link.get("href") or "").strip()
        low = href.lower()
        if not low.startswith("http") or ".h5" not in low:
            continue
        if low.split("?")[0].rsplit("/", 1)[-1] != filename.lower():
            continue
        score = 0
        if "opendap" in low:
            score += 10                       # usable, but not for a plain download
        if "s3credentials" in low or low.startswith("s3://"):
            score += 20
        if "/archive/" in low or "ladsweb" in low:
            score -= 5                        # the classic archive path, to be preferred
        candidates.append((score, href))
    return min(candidates)[1] if candidates else None


# --- 6.3  One request, one window --------------------------------------------
def _cmr_request(product, debut, fin, bbox, tiles=None):
    """Query CMR over one window and return a list of granules."""
    params = {
        "short_name": product,
        "temporal[]": f"{debut}T00:00:00Z,{fin}T23:59:59Z",
        "bounding_box[]": ",".join(f"{v}" for v in bbox),
        "page_size": PAGE_SIZE,
        "sort_key": "start_date",
    }
    rows, page = [], 1

    while page <= MAX_PAGES:
        params["page_num"] = page
        response = None
        for attempt in range(1, 4):                     # 3 attempts, increasing back-off
            try:
                response = SESSION_CMR.get(CMR_URL, params=params, timeout=90)
                response.raise_for_status()
                break
            except Exception as err:
                if attempt == 3:
                    raise RuntimeError(
                        f"The CMR catalogue is unreachable for {product} ({err}).\n"
                        "    -> On Kaggle, switch Internet on in the right-hand panel.\n"
                        "    -> Behind a corporate proxy, export HTTPS_PROXY before launching Jupyter."
                    ) from err
                time.sleep(3 * attempt)

        entries = response.json().get("feed", {}).get("entry", [])
        for e in entries:
            filename = e.get("producer_granule_id") or e.get("title") or ""
            if not filename.lower().endswith(".h5"):
                for link in e.get("links", []):
                    base = (link.get("href") or "").split("?")[0].rsplit("/", 1)[-1]
                    if base.lower().endswith(".h5"):
                        filename = base
                        break
            info = _parse_name(filename)
            if not info:
                continue
            if tiles and info["tile"] not in {t.lower() for t in tiles}:
                continue                                   # tile outside our country
            url = _pick_url(e.get("links"), filename)
            if not url:
                continue
            try:
                size = float(e.get("granule_size") or 0.0)
            except (TypeError, ValueError):
                size = 0.0
            rows.append(dict(filename=filename, url=url, size_mb=round(size, 2), **info))

        if len(entries) < PAGE_SIZE:                     # last page reached
            break
        page += 1
    return rows


CATALOGUE_COLUMNS = ["product", "period", "date", "year", "doy",
                      "tile", "version", "size_mb", "filename", "url"]


def query_cmr(product, years, bbox, tiles=None, trace=None):
    """Walk the product windows, one year after another."""
    rows = []
    for label_html, d0, d1 in windows(product, years):
        trouves = _cmr_request(product, d0, d1, bbox, tiles)
        for g in trouves:
            g["period"] = label_html
        rows.extend(trouves)
        if trace is not None:
            trace.append(dict(period=label_html, granules=len(trouves),
                              tiles=len({g["tile"] for g in trouves})))

    if not rows:
        return pd.DataFrame(columns=CATALOGUE_COLUMNS)

    df = pd.DataFrame(rows)
    # Deduplication: one entry per (tile, date), keeping the highest version
    df = (df.sort_values(["tile", "date", "version"], ascending=[True, True, False])
            .drop_duplicates(subset=["product", "tile", "date"], keep="first")
            .sort_values(["date", "tile"])
            .reset_index(drop=True))
    return df[CATALOGUE_COLUMNS]


# --- 6.4  Preview of the grid, before any request ----------------------------
header("Temporal grid requested")
for p in PRODUCTS_TO_TEST:
    f = windows(p, YEARS)
    preview = ", ".join(e for e, _, _ in f[:4])
    more = f" … {f[-1][0]}" if len(f) > 4 else ""
    field(f"{p} ({PRODUCT_CATALOGUE[p]['frequency']})",
          f"{len(f)} window(s): {preview}{more}", SYM["ok"])
    skipped = len(YEARS) - len(f)
    if skipped:
        print(f"      {SYM['att']} {skipped} year(s) without {FIXED_DAY} "
              f"(29 February outside a leap year)")

print(f"\n  {SYM['pt']} Total: {sum(len(windows(p, YEARS)) for p in PRODUCTS_TO_TEST)} "
      f"catalogue requests, with {len(TILES)} tile(s) expected per window")

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">STEP 07 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Availability test across the four products</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">What NASA actually publishes, year by year</div></div>

This is the heart of the notebook. For each product, it walks through the yearly windows and compares three quantities:

- **expected** = number of windows × number of tiles, what theory predicts;
- **found** = what the catalogue actually returns;
- **completeness** = the ratio of the two.

A gap is almost never a bug, and that is the whole value of the exercise. The annual composite for the current year is not published yet. The daily products start on 19 January 2012. And above all, on a single day, a year can be missing because the night was cloudy — crucial information that a blind download would have hidden.

The table also gives the **total volume** per product, the decisive figure before launching anything. No token is needed at this step.

In [ ]:
# =============================================================================
#  STEP 7 - Probe the products: availability, completeness, volume
# =============================================================================
header(f"Black Marble availability - {COUNTRY_NAME} ({COUNTRY_ISO3})")

inventory, catalogue_parts, traces = [], [], {}
network_down = False          # avoids retrying four times when the network is down

for product in PRODUCTS_TO_TEST:
    grid = windows(product, YEARS)
    expected = len(grid) * len(TILES)
    t0 = time.perf_counter()
    print(f"\n  {SYM['fl']} {product} — {PRODUCT_CATALOGUE[product]['label']}  "
          f"({len(grid)} window(s): {grid[0][0]} -> {grid[-1][0]})")

    trace = []
    if network_down:
        df, error = pd.DataFrame(), "network unavailable (see the previous product)"
        print(f"     {SYM['ko']} {error}")
    else:
        try:
            df = query_cmr(product, YEARS, BBOX, TILES, trace=trace)
            error = None
        except Exception as exc:
            df, error = pd.DataFrame(), str(exc)
            network_down = True
            for l in error.splitlines():
                print(f"     {SYM['ko']} {l.strip()}")

    elapsed = time.perf_counter() - t0
    found = len(df)
    volume = float(df["size_mb"].sum()) if found else 0.0
    periods_ok = sorted(df["period"].unique()) if found else []
    traces[product] = trace

    inventory.append(dict(
        product=product,
        frequency=PRODUCT_CATALOGUE[product]["frequency"],
        grid=f"{grid[0][0]} -> {grid[-1][0]}",
        windows=len(grid), tiles=len(TILES),
        expected=expected, found=found,
        complete_pct=round(100 * found / expected, 1) if expected else 0.0,
        periods_published=len(periods_ok),
        volume_mb=round(volume, 1),
        mb_per_file=round(volume / found, 1) if found else 0.0,
        seconds=round(elapsed, 1),
        error=error or "",
    ))

    if found:
        catalogue_parts.append(df)
        mark = SYM["ok"] if found >= expected else SYM["att"]
        print(f"     {mark} {found} granule(s) out of {expected} expected - "
              f"{len(periods_ok)}/{len(grid)} period(s) published - "
              f"{volume/1024:.2f} GB - {elapsed:.1f} s")
    elif not error:
        print(f"     {SYM['att']} no granule found on this grid.")

INVENTORY = pd.DataFrame(inventory)
CATALOGUE = (pd.concat(catalogue_parts, ignore_index=True)
             if catalogue_parts else pd.DataFrame(columns=CATALOGUE_COLUMNS))

# --- Summary table -----------------------------------------------------------
header("Summary")
_view = INVENTORY[["product", "frequency", "grid", "windows", "tiles", "expected",
                   "found", "complete_pct", "periods_published", "volume_mb"]]
_view = _view.rename(columns={"complete_pct": "complete_%", "volume_mb": "volume_MB",
                            "periods_published": "periods_ok"})
print(_view.to_string(index=False))

# --- Exports -----------------------------------------------------------------
_f1 = FOLDERS["catalogue"] / f"inventory_{COUNTRY_ISO3}.csv"
INVENTORY.to_csv(_f1, index=False, encoding="utf-8-sig")
print(f"\n  {SYM['ok']} Inventory written : {_f1}")
if len(CATALOGUE):
    _f2 = FOLDERS["catalogue"] / f"granules_{COUNTRY_ISO3}.csv"
    CATALOGUE.to_csv(_f2, index=False, encoding="utf-8-sig")
    print(f"  {SYM['ok']} Catalogue written : {_f2}  ({len(CATALOGUE)} granules)")
    print(f"  {SYM['pt']} Total volume if everything were downloaded: "
          f"{CATALOGUE['size_mb'].sum()/1024:.2f} Go")

The detail below answers the question that comes next: **which years are missing, exactly, and for which product?** An incomplete year — fewer tiles than expected — is reported separately from a year that is entirely absent: the first is fixed by shifting the date a few days, the second usually reveals a limit of the product itself.

In [ ]:
# =============================================================================
#  STEP 7b - Coverage, period by period
# =============================================================================
if len(CATALOGUE):
    for product in PRODUCTS_TO_TEST:
        sub = CATALOGUE[CATALOGUE["product"] == product]
        grid = windows(product, YEARS)
        if not len(sub):
            print(f"\n  {product} - {SYM['ko']} no period published")
            continue

        by_period = sub.groupby("period")["tile"].nunique().to_dict()
        expected_labels = [e for e, _, _ in grid]
        complete = [e for e in expected_labels if by_period.get(e, 0) >= len(TILES)]
        partial = [e for e in expected_labels if 0 < by_period.get(e, 0) < len(TILES)]
        missing = [e for e in expected_labels if by_period.get(e, 0) == 0]

        print(f"\n  {product} - {len(complete)}/{len(expected_labels)} complete period(s) "
              f"({len(TILES)} tile(s) each)")
        if partial:
            detail = ", ".join(f"{e} ({by_period[e]}/{len(TILES)})" for e in partial[:8])
            print(f"     {SYM['att']} incomplete: {detail}"
                  + (f" … (+{len(partial)-8})" if len(partial) > 8 else ""))
        if missing:
            preview = ", ".join(missing[:12])
            print(f"     {SYM['ko']} missing   : {preview}"
                  + (f" … (+{len(missing)-12})" if len(missing) > 12 else ""))
        if not partial and not missing:
            print(f"     {SYM['ok']} complete series over the whole requested grid")

    print("\n  Catalogue excerpt (first 6 rows):")
    print(CATALOGUE.head(6)[["product", "period", "date", "tile", "version",
                             "size_mb", "filename"]].to_string(index=False))
else:
    print(f"  {SYM['att']} Empty catalogue - nothing to detail.")

Two graphical readings close the step. The first compares expected with published and quantifies the volume. The second is a **coverage matrix**: one row per product, one column per year, a green cell when all tiles are there. This is the view that supports the decision — if a row is full of holes, that product will not carry a reliable time series for your country, and it is better to know now.

In [ ]:
# =============================================================================
#  STEP 7c - Availability, read graphically
# =============================================================================
if MPL_OK and len(INVENTORY) and INVENTORY["found"].sum() > 0:
    import numpy as np
    import matplotlib.pyplot as plt

    RAMP = ["#00553A", "#00704A", "#00A86A", "#57BD92"]
    data = INVENTORY[INVENTORY["found"] > 0].reset_index(drop=True)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.6, 4.2), dpi=110)

    y = np.arange(len(data))
    ax1.barh(y, data["expected"], color="#E8F5EF", edgecolor="#D5DED9",
             height=.62, label="expected")
    ax1.barh(y, data["found"], color=[RAMP[i % 4] for i in y],
             height=.62, label="published")
    for i, r in data.iterrows():
        ax1.text(max(r["expected"], r["found"]) * 1.03, i,
                 f"{int(r['found'])}/{int(r['expected'])}  ({r['complete_pct']:.0f} %)",
                 va="center", fontsize=8.6, color="#231F20")
    ax1.set_yticks(y, data["product"], fontsize=9.5, fontweight="bold")
    ax1.invert_yaxis()
    ax1.set_xlim(0, data[["expected", "found"]].max().max() * 1.42)
    ax1.set_xlabel("Number of granules", fontsize=9, color="#5E6964")
    ax1.set_title("Granules published on the requested grid",
                  fontsize=10.5, fontweight="bold", color="#231F20", pad=10)
    ax1.legend(fontsize=8, frameon=False, loc="lower right")

    ax2.barh(y, data["volume_mb"] / 1024, color=[RAMP[i % 4] for i in y], height=.62)
    for i, r in data.iterrows():
        ax2.text(r["volume_mb"] / 1024 * 1.03, i,
                 f"{r['volume_mb']/1024:.2f} GB  ·  {r['mb_per_file']:.0f} MB/file",
                 va="center", fontsize=8.6, color="#231F20")
    ax2.set_yticks(y, data["product"], fontsize=9.5, fontweight="bold")
    ax2.invert_yaxis()
    ax2.set_xlim(0, max(data["volume_mb"].max() / 1024, .01) * 1.75)
    ax2.set_xlabel("Volume to download (GB)", fontsize=9, color="#5E6964")
    ax2.set_title("Weight of a full download",
                  fontsize=10.5, fontweight="bold", color="#231F20", pad=10)

    for ax in (ax1, ax2):
        ax.grid(axis="x", color="#E1E7E4", linewidth=.8)
        ax.set_axisbelow(True)
        ax.tick_params(labelsize=8.5, colors="#5E6964")
        for side in ("top", "right", "left"):
            ax.spines[side].set_visible(False)
        ax.spines["bottom"].set_color("#D5DED9")

    fig.suptitle(f"Black Marble - availability for {COUNTRY_NAME} ({len(TILES)} tile(s))",
                 fontsize=12.5, fontweight="bold", color="#00553A", y=1.04)
    fig.tight_layout()
    _fig = FOLDERS["figures"] / f"availability_{COUNTRY_ISO3}.png"
    fig.savefig(_fig, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  {SYM['ok']} Figure saved: {_fig}")
else:
    print(f"  {SYM['att']} Chart skipped (matplotlib missing or empty catalogue).")

In [ ]:
# =============================================================================
#  STEP 7d - Coverage matrix: products x years
# =============================================================================
if MPL_OK and len(CATALOGUE):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap, BoundaryNorm

    products = [p for p in PRODUCTS_TO_TEST
                if len(CATALOGUE[CATALOGUE["product"] == p])]
    matrix = np.zeros((len(products), len(YEARS)))
    texts = [["" for _ in YEARS] for _ in products]

    for i, p in enumerate(products):
        sub = CATALOGUE[CATALOGUE["product"] == p]
        by_period = sub.groupby("period")["tile"].nunique().to_dict()
        for j, a in enumerate(YEARS):
            f = window_for_year(p, a)
            if f is None:                       # date does not exist that year
                matrix[i, j] = -1
                texts[i][j] = "—"
                continue
            n = by_period.get(f[0], 0)
            matrix[i, j] = 2 if n >= len(TILES) else (1 if n else 0)
            texts[i][j] = str(n) if n else ""

    colours = ListedColormap(["#F0F2F1", "#B83B2E", "#F5C242", "#00A86A"])
    norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5], colours.N)

    fig, ax = plt.subplots(figsize=(max(7.5, 0.72 * len(YEARS) + 3.2),
                                    1.05 * len(products) + 2.1), dpi=110)
    ax.imshow(matrix, cmap=colours, norm=norm, aspect="auto")

    for i in range(len(products)):
        for j in range(len(YEARS)):
            if texts[i][j]:
                ax.text(j, i, texts[i][j], ha="center", va="center", fontsize=8.5,
                        color="#FFFFFF" if matrix[i, j] == 2 else "#231F20",
                        fontweight="bold")
    ax.set_xticks(np.arange(len(YEARS)), [str(a) for a in YEARS],
                  fontsize=8.5, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(products)), products, fontsize=9.5, fontweight="bold")
    ax.set_xticks(np.arange(-.5, len(YEARS), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(products), 1), minor=True)
    ax.grid(which="minor", color="#FFFFFF", linewidth=2.4)
    ax.tick_params(which="minor", length=0)
    ax.tick_params(colors="#5E6964")
    for spine in ax.spines.values():
        spine.set_visible(False)

    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=c, edgecolor="#D5DED9")
               for c in ["#00A86A", "#F5C242", "#B83B2E", "#F0F2F1"]]
    ax.legend(handles,
              [f"complete ({len(TILES)} tiles)", "partial", "missing", "no such date"],
              loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=4,
              fontsize=8.5, frameon=False)

    day = f"{int(FIXED_DAY.split('-')[1])} {MONTH_NAME[int(FIXED_DAY.split('-')[0])]}"
    ax.set_title(f"Coverage by year - {COUNTRY_NAME} ({COUNTRY_ISO3})\n"
                 f"daily: {day} of each year  ·  monthly: {MONTH_NAME[FIXED_MONTH]}",
                 fontsize=11.5, fontweight="bold", color="#00553A", pad=14)
    fig.tight_layout()
    _fig = FOLDERS["figures"] / f"coverage_{COUNTRY_ISO3}.png"
    fig.savefig(_fig, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  {SYM['ok']} Figure saved: {_fig}")
    print(f"  {SYM['pt']} The number in each cell is how many tiles were found.")
else:
    print(f"  {SYM['att']} Matrix skipped (matplotlib missing or empty catalogue).")

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">STEP 08 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Download</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Several products, resume on drop, integrity check</div></div>

A scientific download is not a `wget`. Four behaviours are expected of a serious acquisition chain, and the function below implements them all:

| Behaviour | Why |
|---|---|
| **Resume** from a partial file (`Range: bytes=…`) | a drop at 90 % must not cost 90 % of the work |
| **Manual redirect following** | Earthdata redirects lose the `Authorization` header unless it is re-injected |
| **Verification of the announced size** | a truncated file looks perfectly normal on disk |
| **Control opening** in HDF5 | the only proof that an `.h5` is actually readable |

### Choosing what comes down

`PRODUCTS_TO_DOWNLOAD` is a **list**: you can pull the daily, monthly and annual products in a single pass. The mode decides the volume — `"none"` stops at the inventory, `"sample"` takes the most recent files of each product up to `MAX_FILES_PER_PRODUCT`, `"full"` takes the whole grid.

The total volume is announced **before** starting, product by product. Files are filed by country, product and year, and rerunning the cell is safe: whatever is already valid is skipped.

In [ ]:
# =============================================================================
#  STEP 8 - Verified, resumable, idempotent download
# =============================================================================
SESSION_DL = requests.Session()
SESSION_DL.headers.update({"User-Agent": f"{CLIENT_ID}/1.0"})
if TOKEN:
    SESSION_DL.headers.update({"Authorization": f"Bearer {TOKEN}"})


def file_is_valid(path, min_size=1_000_000):
    """The file exists, has a credible size, and really opens as HDF5."""
    if not path.exists() or path.stat().st_size < min_size:
        return False
    if not H5PY_OK:
        return True                      # without h5py, size is the only criterion
    try:
        import h5py
        with h5py.File(path, "r"):
            return True
    except Exception:
        return False


def check_token(test_url):
    """Request the first bytes of a granule: a clear diagnosis before the loop."""
    if not TOKEN:
        return False, "no token provided"
    try:
        r = SESSION_DL.get(test_url, headers={"Range": "bytes=0-2047"},
                           stream=True, timeout=60)
        content_type = r.headers.get("content-type", "").lower()
        if "html" in content_type:
            return False, "HTML response - invalid or expired token, or EULA not accepted"
        if r.status_code in (401, 403):
            return False, f"HTTP {r.status_code} - token refused by Earthdata"
        if r.status_code not in (200, 206):
            return False, f"unexpected HTTP {r.status_code}"
        r.close()
        return True, "token accepted"
    except Exception as exc:
        return False, f"network unavailable ({exc})"


def download(url, destination, attempts=4):
    """Download one granule. Returns ('skipped' | 'ok' | 'failed', bytes)."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(".h5.part")

    if file_is_valid(destination):
        return "skipped", destination.stat().st_size

    for attempt in range(1, attempts + 1):
        try:
            resumed = partial.stat().st_size if partial.exists() else 0
            headers = {"Range": f"bytes={resumed}-"} if resumed else {}

            current, response = url, None
            for _ in range(10):                      # redirects, keeping the token
                response = SESSION_DL.get(current, headers=headers, stream=True,
                                         timeout=300, allow_redirects=False)
                if response.status_code in (301, 302, 303, 307, 308):
                    current = response.headers["Location"]
                    continue
                break

            if "html" in response.headers.get("content-type", "").lower():
                return "failed", 0                   # no point retrying: the token is at fault
            if response.status_code == 416:           # already complete server-side
                partial.replace(destination)
                return "ok", destination.stat().st_size
            if response.status_code not in (200, 206):
                if attempt < attempts:
                    time.sleep(5 * attempt)
                    continue
                return "failed", 0

            expected = int(response.headers.get("content-length", 0))
            if response.status_code == 206:
                expected += resumed
            mode = "ab" if (resumed and response.status_code == 206) else "wb"

            received, milestone = resumed, 0
            with open(partial, mode) as out:
                for chunk in response.iter_content(1 << 20):   # 1 Mo
                    if not chunk:
                        continue
                    out.write(chunk)
                    received += len(chunk)
                    if expected:
                        done = int(100 * received / expected)
                        if done >= milestone + 25:
                            milestone = done - done % 25
                            print(f"        {milestone:3d} %", end="\r")
            if expected:
                print(" " * 24, end="\r")            # clears the progress line

            if expected and partial.stat().st_size < expected:
                raise IOError(f"truncated: {partial.stat().st_size}/{expected} bytes")

            partial.replace(destination)
            if not file_is_valid(destination):
                destination.unlink(missing_ok=True)
                raise IOError("file unreadable as HDF5")
            return "ok", destination.stat().st_size

        except Exception as exc:
            print(f"        {SYM['att']} essai {attempt}/{attempts} : {exc}")
            if attempt < attempts:
                time.sleep(8 * attempt)                # the .part file is kept for resuming

    return "failed", 0


# --- 8.1  Selection: a subset per product ------------------------------------
header("Download")

selections = {}
if DOWNLOAD_MODE != "none" and len(CATALOGUE):
    for product in PRODUCTS_TO_DOWNLOAD:
        available = CATALOGUE[CATALOGUE["product"] == product].copy()
        if not len(available):
            print(f"  {SYM['att']} {product}: no granule in the catalogue, product skipped.")
            continue
        available = available.sort_values(["date", "tile"], ascending=[False, True])
        chosen = available if DOWNLOAD_MODE == "full" else available.head(MAX_FILES_PER_PRODUCT)
        selections[product] = chosen
        field(f"{product}",
              f"{len(chosen)} file(s) out of {len(available)} - "
              f"{chosen['size_mb'].sum()/1024:.2f} GB - "
              f"periods {chosen['period'].min()} -> {chosen['period'].max()}", SYM["ok"])
else:
    print(f"  {SYM['pt']} Mode '{DOWNLOAD_MODE}' - no download requested.")

to_download = (pd.concat(selections.values(), ignore_index=True)
                 if selections else pd.DataFrame())
if len(to_download):
    total_gb = to_download["size_mb"].sum() / 1024
    print()
    field("TOTAL to download", f"{len(to_download)} file(s) - {total_gb:.2f} GB",
          SYM["ok"] if total_gb < 10 else SYM["att"])
    if total_gb > 10:
        print(f"     {SYM['att']} Over 10 GB: check free disk space before going further.")

# --- 8.2  Token check, then the loop -----------------------------------------
manifest = []
if len(to_download):
    token_ok, diagnosis = check_token(to_download.iloc[0]["url"])
    field("Token check", diagnosis, SYM["ok"] if token_ok else SYM["ko"])

    if not token_ok:
        print(f"\n  {SYM['ko']} Download aborted - the inventory is still usable.")
        print("     Check step 3, then rerun this cell alone.")
    else:
        for product, chosen in selections.items():
            print(f"\n  {SYM['fl']} {product} - {len(chosen)} file(s)")
            for rank, (_, g) in enumerate(chosen.iterrows(), 1):
                target = (FOLDERS["h5_cache"] / COUNTRY_ISO3 / g["product"]
                         / str(g["year"]) / g["filename"])
                print(f"    [{rank}/{len(chosen)}] {g['period']} · {g['tile']} · "
                      f"{g['filename']}  ({g['size_mb']:.0f} Mo)")
                state, nbytes = download(g["url"], target)
                symbol = {"ok": SYM["ok"], "skipped": SYM["pt"], "failed": SYM["ko"]}[state]
                word = {"ok": "downloaded", "skipped": "already present", "failed": "FAILED"}[state]
                print(f"        {symbol} {word}" + (f" — {nbytes/1e6:.1f} Mo" if nbytes else ""))
                manifest.append(dict(product=g["product"], period=g["period"],
                                      filename=g["filename"], tile=g["tile"],
                                      date=str(g["date"]), status=state,
                                      nbytes=nbytes, path=str(target)))
                time.sleep(0.3)                      # courtesy towards the server

if manifest:
    MANIFEST = pd.DataFrame(manifest)
    _fm = FOLDERS["catalogue"] / f"manifest_{COUNTRY_ISO3}.csv"
    MANIFEST.to_csv(_fm, index=False, encoding="utf-8-sig")
    succeeded = int((MANIFEST["status"] != "failed").sum())
    print(f"\n  {SYM['ok']} {succeeded}/{len(MANIFEST)} file(s) available on disk")
    for product, sub in MANIFEST.groupby("product"):
        print(f"     {SYM['pt']} {product}: {int((sub['status'] != 'failed').sum())} file(s), "
              f"{sub['nbytes'].sum()/1e9:.2f} Go")
    print(f"  {SYM['ok']} Manifest: {_fm}")
else:
    MANIFEST = pd.DataFrame(columns=["product", "period", "filename", "tile",
                                      "date", "status", "nbytes", "path"])

<div style="border-left:6px solid #00A86A;background:#E8F5EF;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">STEP 09 / 09</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">Quality control</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">Open a granule, read the radiance, look at the image</div></div>

A downloaded file is not yet usable data. Three checks close the collection.

**We do not guess the internal path.** The HDF5 structure differs from one product to the next (`VNP_Grid_DNB`, `VIIRS_Grid_DNB_2d`…). Rather than hard-coding a path that will break at the first change of product, we walk the tree and look the variable up by name.

**We apply the attributes.** Values are stored as integers to save space. Physical radiance is obtained through `value × scale_factor + add_offset`, after masking pixels flagged `_FillValue`. Skipping that step produces maps with absurd orders of magnitude — it is the most common mistake on Black Marble data.

**We look at the image.** The distribution of night-time radiance is extremely skewed: a handful of urban pixels crush everything else. A linear scale gives a black image with three white dots, so we compress the dynamic range before displaying.

In [ ]:
# =============================================================================
#  STEP 9 - Quality control on a downloaded granule
# =============================================================================
local_files = sorted((FOLDERS["h5_cache"] / COUNTRY_ISO3).rglob("*.h5")) \
    if (FOLDERS["h5_cache"] / COUNTRY_ISO3).exists() else []

header("Quality control")

if not local_files:
    print(f"  {SYM['att']} No .h5 file on disk for {COUNTRY_ISO3}.")
    print("     Set DOWNLOAD_MODE to 'sample' in step 4 and rerun step 8.")
elif not H5PY_OK:
    print(f"  {SYM['att']} h5py missing - install it and rerun: pip install h5py")
    for f in local_files[:5]:
        field(f.name, f"{f.stat().st_size/1e6:.1f} MB", SYM["pt"])
else:
    import h5py
    import numpy as np

    SAMPLE = local_files[-1]                 # the most recent by alphabetical order
    field("Files on disk", str(len(local_files)), SYM["ok"])
    field("File examined", SAMPLE.name, SYM["ok"])
    field("Size", f"{SAMPLE.stat().st_size/1e6:.1f} MB", SYM["pt"])

    # --- 9.1  Inventory of variables, without assuming the path ---------------
    variables = {}
    with h5py.File(SAMPLE, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset) and obj.ndim == 2:
                variables[name] = obj.shape
        f.visititems(visit)

        print(f"\n  Two-dimensional variables ({len(variables)}):")
        for name, shape in list(variables.items())[:14]:
            print(f"     {SYM['pt']} {name.split('/')[-1]:<44} {shape}")
        if len(variables) > 14:
            print(f"     ... and {len(variables)-14} more")

        # --- 9.2  Reading the product's main variable ------------------------
        info = _parse_name(SAMPLE.name) or {}
        wanted = PRODUCT_CATALOGUE.get(info.get("product", ""), {}).get("variable", "")
        path = next((n for n in variables if n.endswith(wanted)), None) \
            or next((n for n in variables
                     if any(c in n for c in ("NTL", "Radiance", "Composite"))), None)

        if path is None:
            print(f"\n  {SYM['att']} No radiance variable found in this file.")
            array = None
        else:
            def read_attr(value):
                """An HDF5 attribute can be an array, a scalar or raw bytes."""
                if isinstance(value, bytes):
                    return value.decode("utf-8", "ignore")
                flat = np.ravel(value)
                if flat.size == 1:
                    v = flat[0]
                    return v.decode("utf-8", "ignore") if isinstance(v, bytes) else v
                return value

            table = f[path]
            raw = table[:].astype("float64")
            attrs = {k: table.attrs[k] for k in table.attrs}
            scale = float(read_attr(attrs.get("scale_factor", 1.0)))
            offset = float(read_attr(attrs.get("add_offset", 0.0)))
            fill = attrs.get("_FillValue", None)

            array = raw.copy()
            if fill is not None:
                array[raw == float(read_attr(fill))] = np.nan
            array = array * scale + offset

            print(f"\n  Variable selected: {path.split('/')[-1]}")
            field("  internal path", path, SYM["pt"])
            field("  dimensions", f"{table.shape[0]} × {table.shape[1]} pixels "
                                  f"(~{table.shape[0]*table.shape[1]/1e6:.1f} M)", SYM["pt"])
            field("  scale_factor / offset", f"{scale:.6g} / {offset:.6g}", SYM["pt"])
            field("  unit", str(read_attr(attrs.get("units", "nW/cm2/sr"))), SYM["pt"])

            valid = array[np.isfinite(array)]
            if valid.size:
                p50, p95, p99, p999 = np.percentile(valid, [50, 95, 99, 99.9])
                field("  valid pixels", f"{valid.size:,}"
                                          + f"  ({100*valid.size/array.size:.1f} %)", SYM["ok"])
                field("  median / p95", f"{p50:.3f} / {p95:.2f} nW/cm2/sr", SYM["pt"])
                field("  p99 / p99.9 / max", f"{p99:.1f} / {p999:.1f} / {valid.max():.1f}", SYM["pt"])
                field("  dark pixels (<0.5)",
                      f"{100*(valid < 0.5).mean():.1f} % of the tile", SYM["pt"])

In [ ]:
# =============================================================================
#  STEP 9b - Map preview of the tile
# =============================================================================
if local_files and H5PY_OK and MPL_OK and 'array' in globals() and array is not None:
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap, PowerNorm

    # Institutional palette: deep night -> green -> gold -> warm white
    NTL_PALETTE = LinearSegmentedColormap.from_list(
        "bad_ntl", ["#00100B", "#00553A", "#00A86A", "#F5C242", "#FFF6DC"])

    # Downsampling: 2400x2400 pixels add nothing on screen
    step = max(1, array.shape[0] // 1200)
    view = array[::step, ::step]
    ceiling = float(np.nanpercentile(view, 99.5)) or 1.0

    fig, (axa, axb) = plt.subplots(1, 2, figsize=(12.2, 5.2), dpi=110,
                                   gridspec_kw={"width_ratios": [1.25, 1]})

    img = axa.imshow(np.nan_to_num(view), cmap=NTL_PALETTE,
                       norm=PowerNorm(gamma=0.32, vmin=0, vmax=ceiling),
                       interpolation="nearest")
    axa.set_xticks([]); axa.set_yticks([])
    axa.set_title(f"{SAMPLE.name.split('.')[0]} · {info.get('tile','')} · "
                  f"{info.get('date','')}",
                  fontsize=10.5, fontweight="bold", color="#231F20", pad=10)
    cbar = fig.colorbar(img, ax=axa, fraction=.046, pad=.02)
    cbar.set_label("Radiance (nW/cm2/sr) - compressed scale", fontsize=8.5, color="#5E6964")
    cbar.ax.tick_params(labelsize=7.5, colors="#5E6964")

    valid = view[np.isfinite(view) & (view > 0)]
    axb.hist(np.log10(valid + 0.01), bins=90, color="#00A86A", edgecolor="none")
    axb.set_xlabel("log10(radiance + 0.01)", fontsize=9, color="#5E6964")
    axb.set_ylabel("Number of pixels", fontsize=9, color="#5E6964")
    axb.set_title("Distribution: the skew that forces a logarithmic scale",
                  fontsize=10.5, fontweight="bold", color="#231F20", pad=10)
    axb.grid(axis="y", color="#E1E7E4", linewidth=.8); axb.set_axisbelow(True)
    axb.tick_params(labelsize=8.5, colors="#5E6964")
    for side in ("top", "right"):
        axb.spines[side].set_visible(False)
    for side in ("bottom", "left"):
        axb.spines[side].set_color("#D5DED9")

    fig.suptitle(f"Control preview - {COUNTRY_NAME} ({COUNTRY_ISO3})",
                 fontsize=12.5, fontweight="bold", color="#00553A", y=1.02)
    fig.tight_layout()
    _fig = FOLDERS["figures"] / f"preview_{COUNTRY_ISO3}_{info.get('tile','tile')}.png"
    fig.savefig(_fig, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  {SYM['ok']} Figure saved: {_fig}")
    print(f"  {SYM['pt']} A whole 10x10 deg tile: clipping to administrative")
    print("     boundaries happens in the next notebook ('Explore and understand').")
else:
    print(f"  {SYM['att']} Preview skipped (no granule read, or matplotlib/h5py missing).")

<div style="border-left:6px solid #F5C242;background:#FFFDF6;border-radius:0 14px 14px 0;padding:16px 22px;margin:26px 0 10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><div style="color:#D49A00;font-weight:700;font-size:12px;letter-spacing:2.5px;">WRAP-UP</div><div style="color:#00553A;font-size:1.32em;font-weight:800;margin-top:3px;">What the session produced</div><div style="color:#231F20;font-size:.97em;margin-top:4px;">A traceable summary, and a session file for the GitHub repository</div></div>

In [ ]:
# =============================================================================
#  WRAP-UP - Inventory of outputs and session log
# =============================================================================
session_seconds = (dt.datetime.now() - SESSION_START).total_seconds()

header("Session wrap-up")
field("Country", f"{COUNTRY_NAME} ({COUNTRY_ISO3})", SYM["ok"])
field("Tiles", f"{len(TILES)} — {', '.join(TILES)}", SYM["pt"])
field("Products tested", ", ".join(PRODUCTS_TO_TEST), SYM["pt"])
field("Granules catalogued", f"{len(CATALOGUE)}", SYM["ok"] if len(CATALOGUE) else SYM["att"])
field("Granules downloaded", f"{int((MANIFEST['status'] != 'failed').sum()) if len(MANIFEST) else 0}",
      SYM["pt"])
field("Total duration", f"{session_seconds:.0f} s", SYM["pt"])

print("\n  Files produced:")
total = 0
for folder in ("catalogue", "figures"):
    for f in sorted(FOLDERS[folder].glob("*")):
        if f.is_file():
            total += f.stat().st_size
            print(f"     {SYM['pt']} {folder}/{f.name:<44} {f.stat().st_size/1024:8.1f} kB")
for f in sorted((FOLDERS["h5_cache"] / COUNTRY_ISO3).rglob("*.h5")) if (FOLDERS["h5_cache"] / COUNTRY_ISO3).exists() else []:
    total += f.stat().st_size
    print(f"     {SYM['pt']} h5_cache/…/{f.name:<38} {f.stat().st_size/1e6:8.1f} MB")
print(f"\n     Total written: {total/1e6:.1f} MB  in  {ROOT}")

# Session log: reproducibility and traceability for the GitHub repository
session = dict(
    horodatage=SESSION_START.isoformat(timespec="seconds"),
    duree_s=round(session_seconds, 1),
    platform=PLATFORM, python=platform.python_version(),
    country=dict(iso3=COUNTRY_ISO3, name=COUNTRY_NAME, bbox=BBOX, tiles=TILES),
    products=PRODUCTS_TO_TEST,
    grid=dict(years=YEARS, jour_fixe=FIXED_DAY, mois_fixe=FIXED_MONTH),
    mode_telechargement=DOWNLOAD_MODE,
    products_downloaded=PRODUCTS_TO_DOWNLOAD,
    inventory=INVENTORY.drop(columns=["error"]).to_dict("records"),
    racine=str(ROOT),
)
_log_file = FOLDERS["logs"] / f"session_{COUNTRY_ISO3}_{SESSION_START:%Y%m%d_%H%M%S}.json"
_log_file.write_text(json.dumps(session, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(f"\n  {SYM['ok']} Session log: {_log_file}")
print(f"\n  {SYM['fl']} Next step: the 'Explore and understand' notebook - quality masks,")
print("     zonal statistics by administrative region, time series.")

---

## Troubleshooting

| Symptom | Most likely cause | Fix |
|---|---|---|
| `ISO3 code is not in the directory` | alpha-2 code (`CI`) instead of alpha-3 (`CIV`) | use the three-letter code, or set `CUSTOM_BBOX` |
| Empty catalogue for every product | no outbound network access | on Kaggle, switch *Internet* on in the right-hand panel; behind a corporate proxy, export `HTTPS_PROXY` |
| `HTML response - invalid token` | token expired, truncated on copy, or EULA not accepted | regenerate the token on the Earthdata profile page and accept the *LAADS DAAC* EULA |
| HTTP 401 / 403 on download | token lost from the header after a redirect | already handled here; if it persists, the token is being refused — regenerate it |
| The most recent annual composite is missing | it is not published yet | normal behaviour: annual composites appear with several months of delay |
| Missing nights in `VNP46A1` / `A2` | these products start on 19 January 2012 | adjust the window, or accept the gap and document it |
| Interrupted download | network drop | rerun the cell: the `.part` file resumes where it stopped |
| `file unreadable as HDF5` | truncated download that went undetected | the file is deleted automatically; rerun |
| Out of space on Colab | session disk full | mount Google Drive and redefine `NTL_HOME` |
| Entirely black map | `scale_factor` not applied, or linear scale | check step 9.2: the scale must be compressed |
| `UnicodeEncodeError` in a Windows console | legacy cp1252 console | already handled; otherwise run `chcp 65001` before starting Python |


<div style="background:#E8F5EF;border:1px solid #00704A44;border-radius:12px;padding:13px 18px;margin:10px 0;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;"><b style="color:#00704A;">ℹ️ What this notebook does not do — and where it is done</b><div style="color:#231F20;margin-top:5px;line-height:1.55;">Collection stops at the raw tile. Clipping to national and regional boundaries, applying the quality masks, zonal statistics and validation against official statistics belong to parts 2 to 4 of Day 4. A Black Marble tile is a 10-degree square&nbsp;: it overshoots your country by a wide margin, and that is normal.</div></div>

---

## References and data licence

**Products and documentation**

- Román, M. O. *et al.* (2018). *NASA's Black Marble nighttime lights product suite*. **Remote Sensing of Environment**, 210, 113–143. <https://doi.org/10.1016/j.rse.2018.03.017>
- *Black Marble User Guide*, NASA LAADS DAAC — <https://ladsweb.modaps.eosdis.nasa.gov>
- Product DOIs: `10.5067/VIIRS/VNP46A1.001` · `…VNP46A2.001` · `…VNP46A3.001` · `…VNP46A4.001`

**Services used**

- NASA CMR — granule search: <https://cmr.earthdata.nasa.gov/search/granules.json>
- Earthdata Login — accounts and tokens: <https://urs.earthdata.nasa.gov>

**National extents**

- Bounding boxes derived from Natural Earth (public domain). For query purposes only: these are not official boundaries and they imply no position on the delineation of any border.

**Data licence**

Black Marble products fall under NASA's open data policy: free reuse, including commercial, subject to citation. For an official publication, cite the product, its version, its DOI and the extraction date — all of which the session log records for you.

**Suggested citation for this notebook**

> African Development Bank (2026). *Night-Time Lights: data collection from NASA Black Marble*. Specialized Technical Group No. 17 — Emerging Issues, "Emerging Issues, Emerging Practice" workshop. Notebook, Day 4, part 1.


<div style="background:linear-gradient(135deg,#00553A 0%,#00704A 60%,#00A86A 100%);border-radius:18px;padding:26px 34px;margin-top:22px;font-family:'Segoe UI', Calibri, system-ui, -apple-system, sans-serif;text-align:center;"><div style="color:#F5C242;font-size:11.5px;letter-spacing:3px;font-weight:700;">DATA TOOLKIT — AFRICAN DEVELOPMENT BANK</div><div style="color:#fff;font-size:1.18em;font-weight:700;margin-top:8px;">One ISO3 code in &nbsp;·&nbsp; four products tested &nbsp;·&nbsp; a reproducible inventory out</div><div style="color:#CFEEDE;font-size:.94em;margin-top:10px;">Day 4 &nbsp;·&nbsp; Part 1 &ldquo;Collect&rdquo; &nbsp;→&nbsp; Part 2 &ldquo;Explore and understand&rdquo;</div><div style="margin:16px auto 0 auto;height:3px;width:110px;background:#F5C242;border-radius:2px;"></div></div>